# MMDENTAL DATASET PRE-PROCESSING


## 1. ENVIRONMENT AND DRIVE SETUP

In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
!pip install -q SimpleITK nibabel monai einops tqdm scikit-learn
print("dependencies installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 MB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 61.4 MB/s eta 0:00:00
dependencies installed


## 2. CONFIGURATION


In [3]:
# ---- stdlib ----
import dataclasses
import gc
import hashlib
import html
import json
import logging
import os
import pathlib
import re
import unicodedata

# ---- third-party ----
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.ndimage
import SimpleITK as sitk
import sklearn.model_selection
import tqdm.auto

SEED = int(os.getenv("MMD_SEED", "42"))
np.random.seed(SEED)

logging.basicConfig(level=logging.INFO,
                    format='%(asctime)s [%(levelname)s] %(message)s', datefmt='%H:%M:%S')
log = logging.getLogger('mmd_preproc')

PROJECT_ROOT = pathlib.Path(os.getenv(
    "DENTAL_PROJECT_ROOT", "/content/drive/MyDrive/Teeth_Segmentation_Classification"))
MMDENTAL_RAW = PROJECT_ROOT / 'MMDental'

MMD_PREPROC          = PROJECT_ROOT / 'Preprocessed_Dataset' / 'MMDental'
MMD_IMG3D_DIR        = MMD_PREPROC / 'images_3d'              # classifier-space
MMD_TEACHER_IMG_DIR  = MMD_PREPROC / 'images_teacher_space'   # teacher-space
MMD_GEOMETRY_DIR     = MMD_PREPROC / 'teacher_geometry'
MMD_MANIFEST_DIR     = MMD_PREPROC / 'manifests'
MMD_QA_DIR           = MMD_PREPROC / 'qa'
MMD_LOG_DIR          = MMD_PREPROC / 'logs'

MMD_MANIFEST_CSV     = MMD_MANIFEST_DIR / 'manifest.csv'
MMD_SPLITS_CSV       = MMD_MANIFEST_DIR / 'splits.csv'
MMD_CV_CSV           = MMD_MANIFEST_DIR / 'cv_assignments.csv'
MMD_QA_JSONL         = MMD_QA_DIR / 'qa_pairs.jsonl'
MMD_LABELS_CSV       = MMD_LOG_DIR / 'mmd_patient_labels.csv'

# Biomarker Table
STAGE_REST_ROOT      = PROJECT_ROOT / 'Stage2_to_Stage6_Biomarker_Arbiter_QA'
PER_PATIENT_BIOMARKER_CSV = STAGE_REST_ROOT / 'biomarkers' / 'per_patient_biomarkers.csv'

for d in [MMD_PREPROC, MMD_IMG3D_DIR, MMD_TEACHER_IMG_DIR, MMD_GEOMETRY_DIR,
          MMD_MANIFEST_DIR, MMD_QA_DIR, MMD_LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)


@dataclasses.dataclass
class MMDConfig:
    binary_class_names: tuple = ('NON_PERIODONTITIS', 'PERIODONTITIS')

    # dual-space contract
    classifier_target_shape: tuple = (112, 176, 176)        # D, H, W
    classifier_target_spacing: tuple = (1.0, 1.0, 1.0)      # mm isotropic
    teacher_target_spacing: tuple = (0.4, 0.4, 0.4)         # mm, matches ToothFairy2 / teacher

    # intensity
    fixed_clip_range: tuple = (-1000.0, 3000.0)
    use_percentile_clip: bool = True
    clip_percentiles: tuple = (0.5, 99.5)
    save_dtype_image: str = 'float32'

    # cross-validation
    cv_n_splits: int = 5
    cv_n_repeats: int = 5

    # eligibility
    pediatric_age_max: float = 18.0

    skip_existing: bool = False
    text_max_chars: int = 800
    config_version: str = 'mmd_preproc_v3'


CFG = MMDConfig()
# NOTE: the previous 0.35 / 0.70 / 1.00 sample-weight scheme is removed. Reliability
# tiers (1.00 / 0.75 / 0.50 / 0.25) now come from the label module and match the methodology.
print("config:", CFG.config_version)
print(f"  classifier space: shape {CFG.classifier_target_shape} @ {CFG.classifier_target_spacing} mm")
print(f"  teacher space:    {CFG.teacher_target_spacing} mm isotropic")
print(f"  CBCT raw: {MMDENTAL_RAW}  {'OK' if MMDENTAL_RAW.exists() else 'MISSING'}")


config: mmd_preproc_v3
  classifier space: shape (112, 176, 176) @ (1.0, 1.0, 1.0) mm
  teacher space:    (0.4, 0.4, 0.4) mm isotropic
  CBCT raw: /content/drive/MyDrive/Teeth_Segmentation_Classification/MMDental  OK


## 3. FILE-CONTRACT VALIDATION


In [4]:
def assert_inputs():
    records_csv = MMDENTAL_RAW / 'medical_records.csv'
    problems = []
    if not records_csv.exists():
        problems.append(f"missing clinical records: {records_csv}")
    if not MMDENTAL_RAW.exists():
        problems.append(f"missing MMDental raw dir: {MMDENTAL_RAW}")
    if problems:
        raise FileNotFoundError("input-contract failure:\n  " + "\n  ".join(problems))
    print("inputs present:")
    print(f"  records: {records_csv}")
    n_nii = len(list(MMDENTAL_RAW.glob('*/*.nii.gz')))
    print(f"  CBCT volumes discoverable: {n_nii} (.nii.gz)  "
          + ("OK" if n_nii else "NONE FOUND — image preprocessing will be skipped"))
    return records_csv, n_nii

RECORDS_CSV, N_NII_AVAILABLE = assert_inputs()
HAVE_IMAGES = N_NII_AVAILABLE > 0


inputs present:
  records: /content/drive/MyDrive/Teeth_Segmentation_Classification/MMDental/medical_records.csv
  CBCT volumes discoverable: 403 (.nii.gz)  OK


## 4. CLINICAL-RECORD LOADING AND CLEANING


In [5]:
MMD_TEXT_COLUMNS = ('Sex', 'Main appeal', 'Subsequent', 'Present medical history',
                    'Past medical history', 'Oral Check', 'Diagnosis', 'Treatment plan',
                    'Handle', 'Doctor advices')
MMD_REQUIRED_COLUMNS = ('Filename',) + MMD_TEXT_COLUMNS + ('Age',)
MISSING_TEXT_VALUES = {'', 'nan', 'none', 'null', 'n/a', 'na', 'nil'}
WHITESPACE_RE = re.compile(r'\s+')


def normalise_patient_id(value) -> str:
    if pd.isna(value):
        return ''
    text = str(value).strip()
    if text.lower() in MISSING_TEXT_VALUES:
        return ''
    try:
        number = float(text)
        if number.is_integer():
            return str(int(number))
    except (ValueError, TypeError):
        pass
    return text


def clean_medical_text(value):
    if pd.isna(value):
        return None
    text = unicodedata.normalize('NFKC', str(value))
    text = html.unescape(text)
    text = WHITESPACE_RE.sub(' ', text).strip()
    return None if text.lower() in MISSING_TEXT_VALUES else text


def safe_join(values, sep=' ||VISIT|| '):
    seen, out = set(), []
    for v in values:
        t = clean_medical_text(v)
        if not t or t.lower() in seen:
            continue
        out.append(t); seen.add(t.lower())
    return sep.join(out)


def load_and_clean_mmdental_records(records_path):
    records = pd.read_csv(records_path, encoding='utf-8-sig')   # UTF-8 BOM
    missing = [c for c in MMD_REQUIRED_COLUMNS if c not in records.columns]
    if missing:
        raise ValueError(f"medical_records.csv missing columns: {missing}")
    raw_rows, raw_patients = len(records), records['Filename'].nunique()

    records = records.copy()
    records['patient_id'] = records['Filename'].map(normalise_patient_id)
    records = records[records['patient_id'] != ''].copy()
    records['Age'] = pd.to_numeric(records['Age'], errors='coerce')
    for c in MMD_TEXT_COLUMNS:
        records[c] = records[c].apply(clean_medical_text)

    n_dups = int(records.duplicated(subset=list(records.columns)).sum())
    if n_dups:
        log.warning(f"dropping {n_dups} exact duplicate rows")
        records = records.drop_duplicates(subset=list(records.columns)).reset_index(drop=True)
    records['visit_idx'] = records.groupby('patient_id').cumcount()

    patient_rows = []
    for pid, g in records.groupby('patient_id', sort=True):
        ages = g['Age'].dropna(); sexes = g['Sex'].dropna()
        row = {'patient_id': pid, 'n_visits': len(g),
               'sex': sexes.mode().iloc[0] if len(sexes) else None,
               'age_median': float(ages.median()) if len(ages) else np.nan}
        for c in MMD_TEXT_COLUMNS:
            row[f'{c}_all'] = safe_join(g[c].tolist())
        patient_rows.append(row)
    patient_records = pd.DataFrame(patient_rows)

    records.to_csv(MMD_LOG_DIR / 'mmd_records_clean_row_level.csv', index=False)
    patient_records.to_csv(MMD_LOG_DIR / 'mmd_records_clean_patient_level.csv', index=False)

    print(f"raw rows {raw_rows} | clean rows {len(records)} | patients {patient_records['patient_id'].nunique()}"
          f" | exact dup rows {n_dups}")
    return records, patient_records


records_clean, patient_records_clean = load_and_clean_mmdental_records(RECORDS_CSV)
display(records_clean.head(3))


raw rows 2124 | clean rows 2112 | patients 660 | exact dup rows 12


,Filename,Sex,Age,Main appeal,Subsequent,Present medical history,Past medical history,Oral Check,Diagnosis,Treatment plan,Handle,Doctor advices,patient_id,visit_idx
0,1,female,20.0,"Protruding teeth, consultation for correction",None,The patient complained that he had undergone o...,None,Convex type. Brackets are visible in the upper...,Malformed teeth (K07.302),"The maxillary gap is closed, and the anterior ...","Explain the condition, inform the patient, ask...",Make a follow-up appointment one week later fo...,1,0
1,2,male,22.0,None,"Follow-up visit, no discomfort",None,Nothing special,"*14 Temporarily sealed, no pain when percussion.",None,None,"*14 Remove the temporary seal, rinse and blot ...",None,2,0
2,2,male,22.0,Right tooth pain for several days,None,The patient complained of tooth pain on the ri...,Nothing special,*14 There is a deep carious cavity and dental ...,*18 impacted teeth *14 chronic pulpitis,*18 extractions *14 root canal treatments,*18 Local disinfection with povidone-iodine an...,"Strictly abide by the ""Precautions after Tooth...",2,1


## 5. PATIENT-LEVEL LABEL CONSTRUCTION


In [6]:
"""
build_perio_labels.py  (v2)

Patient-level periodontitis label construction for MMDental.

Single source of truth for the weak record-derived labels consumed by the
MMDental preprocessing notebook, Stage-0 decision gates, and the future
Stage-II classifier. Evidence is taken ONLY from the `Diagnosis` field
(other clinical fields are deliberately not used to set the periodontitis
label). Disease separation is driven by ICD-10 (Chinese clinical extension)
code prefixes where a parenthetical code exists, and by clause-scoped,
negation-aware, apical-guarded phrase rules for code-less rows.

Three-state internal labelling (POSITIVE / NEGATIVE / UNCERTAIN) with a binary
external task, plus audit subgroups, a multi-label evidence vector, an
auxiliary apical target, and the methodology's four reliability tiers.

These are weak, record-derived labels. A NEGATIVE here is a record-negative
(a documented non-periodontal diagnosis), NEVER a radiographically confirmed
negative.
"""

import argparse
import os
import re
import sys

import pandas as pd


DIAGNOSIS_COL = "Diagnosis"
ID_COL = "Filename"


# -----------------------------------------------------------------------------#
# ICD code prefixes (Chinese ICD-10 clinical extension, e.g. K05.300, K04.500) #
# -----------------------------------------------------------------------------#
PERIODONTAL_STRICT_PREFIXES = ("K05.3",)                 # chronic periodontitis -> strict positive
PERIODONTAL_OTHER_PREFIXES = ("K05.6", "K05.4")          # periodontal disease / degeneration
PERIODONTAL_OTHER_EXACT = ("K05.201",)                   # periodontal abscess (other-periodontal, B only)
PERICORONAL_EXACT = ("K05.202", "K05.204")               # pericoronal abscess / pericoronitis (NOT periodontitis)
GINGIVITIS_PREFIXES = ("K05.1",)                          # gingivitis (no attachment loss)
APICAL_PREFIXES = ("K04.2", "K04.35", "K04.4", "K04.5",  # peri-apical / endodontic
                   "K04.6", "K04.7", "K04.8")
PERIO_ENDO_EXACT = ("K04.902", "K05.501")                 # combined perio-endodontic


# --------------------------------------------------------------------------- #
# Phrase lexicon (lowercased; applied per clause with per-match negation)     #
# --------------------------------------------------------------------------- #
PERIO_ENDO_PHRASES = (
    "periodontal pulp syndrome",
    "periodontal-pulp",
    "combined periodontal and pulpal",
    "combined periodontal-pulpal",
    "periodontal-endodontic",
    "endodontic-periodontal",
    "endo-perio",
    "perio-endo",
)

# Affirmative marginal-periodontal wording -> periodontal_present.
PERIODONTAL_PHRASES = (
    "chronic periodontitis",
    "generalized periodontitis",
    "generalised periodontitis",
    "aggressive periodontitis",
    "marginal periodontitis",
)

# Apex-centred wording -> apical_present, NEVER periodontal_present.
APICAL_PHRASES = (
    "apical periodontitis",
    "periapical periodontitis",
    "peri-apical periodontitis",
    "periradicular periodontitis",
    "apical lesion",
    "apical abscess",
    "periapical abscess",
    "apical granuloma",
    "periapical granuloma",
    "apical cyst",
    "periapical",
    "periradicular",
)

OTHER_PERIODONTAL_PHRASES = (
    "periodontal disease",
    "periodontal abscess",
    "periodontal degeneration",
    "periodontal atrophy",
)

PERICORONAL_PHRASES = (
    "pericoronitis",
    "pericoronal abscess",
)

GINGIVITIS_PHRASES = (
    "gingivitis",
)

# Tooth markers can be glued to the diagnosis word ("*46Chronic periodontitis"),
# so substring search is used; the apical guard handles "*46Chronic apical periodontitis".
PERIODONTITIS_RE = re.compile(r"periodontitis")
APICAL_GUARD_RE = re.compile(r"peri[\s\-]?apical|periradicular|apical")

REFERENCE_RE = re.compile(
    r"\b(same as (the )?(initial|above|previous|last)"
    r"( diagnosis(es)?| visit| consultation| treatment)?"
    r"|as (above|before)|see (the )?(initial|above|previous)|ditto|no change)\b"
)
CLAUSE_DELIM_RE = re.compile(r"[,.;:\n]| \|\|visit\|\| ")
NEGATION_RE = re.compile(
    r"(\bno\b|\bnot\b|\bwithout\b|\bnone\b|\bnil\b|\bnever\b|\babsent\b|\bnegative\b|"
    r"\bunremarkable\b|\bdenies?\b|\bdenied\b|no obvious|no abnormal|"
    r"not (reached|detected|found|seen|obvious|present)|free of|absence of|\(-\)|\(–\)| - )"
)

CODE_RE = re.compile(
    r"(?<![A-Za-z0-9])"
    r"([A-Za-z]{1,3}\d{2}(?:\.\d+)?(?:x\d+)?(?:\.\d+)?)"
    r"(?![A-Za-z0-9])"
)


# --------------------------------------------------------------------------- #
# Evidence flags                                                              #
# --------------------------------------------------------------------------- #
def _empty_flags():
    return {
        "periodontal": False,
        "other_periodontal": False,
        "apical": False,
        "gingivitis": False,
        "perio_endo": False,
        "pericoronal": False,
        "pulpal": False,
        "nonperio_code": False,
    }


def extract_codes(text):
    """Return uppercased ICD/local codes from diagnosis text."""
    if not isinstance(text, str):
        return []

    return [
        code.upper()
        for code in CODE_RE.findall(text)
    ]


def _starts_with_any(code, prefixes):
    return any(code.startswith(p) for p in prefixes)


def classify_codes(codes):
    """Resolve ICD codes into evidence flags (high reliability)."""
    f = _empty_flags()
    for c in codes:
        if c in PERIO_ENDO_EXACT:
            f["perio_endo"] = True
        elif _starts_with_any(c, PERIODONTAL_STRICT_PREFIXES):
            f["periodontal"] = True
        elif c in PERIODONTAL_OTHER_EXACT or _starts_with_any(c, PERIODONTAL_OTHER_PREFIXES):
            f["other_periodontal"] = True
        elif c in PERICORONAL_EXACT:
            f["pericoronal"] = True
        elif _starts_with_any(c, GINGIVITIS_PREFIXES):
            f["gingivitis"] = True
        elif _starts_with_any(c, APICAL_PREFIXES):
            f["apical"] = True
        elif c.startswith(("K04.0", "K04.1")):
            f["pulpal"] = True
            f["nonperio_code"] = True
        else:
            # caries, impacted, tooth loss, exam, K05.2-other, Z*, LC*, etc.
            f["nonperio_code"] = True
    return f


def _clause_bounds(text, pos):
    left = 0
    for match in CLAUSE_DELIM_RE.finditer(text, 0, pos):
        left = match.end()

    right_match = CLAUSE_DELIM_RE.search(text, pos)
    right = right_match.start() if right_match else len(text)

    return left, right


def _is_negated(text, pos, window_chars=50):
    left, _ = _clause_bounds(text, pos)
    prefix = text[max(left, pos - window_chars):pos]
    return bool(NEGATION_RE.search(prefix))


def _has_affirmed(text, phrases):
    """True if any phrase appears in a non-negated clause."""
    for ph in phrases:
        for m in re.finditer(re.escape(ph), text):
            if not _is_negated(text, m.start()):
                return True
    return False


def classify_phrases(text):
    """Map free-text diagnosis content to evidence flags with scoped negation."""
    f = _empty_flags()
    t = text.lower()

    # Perio-endo first; strip so it cannot trip the periodontal rule.
    if _has_affirmed(t, PERIO_ENDO_PHRASES):
        f["perio_endo"] = True
    t_stripped = t
    for ph in PERIO_ENDO_PHRASES:
        t_stripped = t_stripped.replace(ph, " perioendo ")

    if _has_affirmed(t_stripped, APICAL_PHRASES):
        f["apical"] = True

    # Explicit affirmative periodontal phrases.
    if _has_affirmed(t_stripped, PERIODONTAL_PHRASES):
        f["periodontal"] = True

    # Bare "periodontitis": periodontal only if not apical-qualified and not negated.
    for m in PERIODONTITIS_RE.finditer(t_stripped):
        pre = t_stripped[max(0, m.start() - 14):m.start()]
        if APICAL_GUARD_RE.search(pre):
            continue
        if _is_negated(t_stripped, m.start()):
            continue
        f["periodontal"] = True
        break

    if _has_affirmed(t_stripped, OTHER_PERIODONTAL_PHRASES):
        f["other_periodontal"] = True
    if _has_affirmed(t_stripped, PERICORONAL_PHRASES):
        f["pericoronal"] = True
    if _has_affirmed(t_stripped, GINGIVITIS_PHRASES):
        f["gingivitis"] = True

    return f


def is_substantive(text):
    """True if a diagnosis cell carries real content (not empty, not a pure back-reference)."""
    if not isinstance(text, str):
        return False
    low = text.lower()
    if REFERENCE_RE.search(low):
        residual = REFERENCE_RE.sub(" ", low)
        if not re.search(r"[a-z]{3,}", residual):
            return False
    return bool(re.search(r"[a-z]{3,}", low))


def visit_evidence(text):
    """Combined per-visit evidence from codes + phrases, plus substantive/codes."""
    f = _empty_flags()
    if not isinstance(text, str):
        return f, False, [], _empty_flags(), _empty_flags()
    codes = extract_codes(text)
    cf = classify_codes(codes)
    pf = classify_phrases(text)
    for k in f:
        f[k] = cf[k] or pf[k]
    return f, is_substantive(text), codes, cf, pf


def aggregate_patient(visit_texts):
    """Union per-visit evidence across all of a patient's visits."""
    agg = _empty_flags()
    has_substantive = False
    perio_from_code = False
    perio_from_phrase = False
    codes_found = []
    matched = set()

    for text in visit_texts:
        f, substantive, codes, cf, pf = visit_evidence(text)
        for k in agg:
            agg[k] = agg[k] or f[k]
        has_substantive = has_substantive or substantive
        codes_found.extend(codes)
        if cf["periodontal"]:
            perio_from_code = True
        elif pf["periodontal"]:
            perio_from_phrase = True
        if isinstance(text, str):
            for k, v in f.items():
                if v:
                    matched.add(k)

    return agg, has_substantive, perio_from_code, perio_from_phrase, codes_found, matched


def assign_labels(agg, has_signal, perio_from_code, perio_from_phrase, has_any_code):
    """Derive internal state, A/B/C membership, aux target, tier, and subgroup.

    `has_signal` is True when the patient has substantive diagnosis text OR any code.
    """
    periodontal = agg["periodontal"]
    other_perio = agg["other_periodontal"]
    apical = agg["apical"]
    gingivitis = agg["gingivitis"]
    perio_endo = agg["perio_endo"]
    pericoronal = agg["pericoronal"]

    phrase_only_perio_endo = perio_endo and not perio_from_code

    if phrase_only_perio_endo:
        internal = "UNCERTAIN"
        tier = 0.25
        source = "perio_endo_ambiguous"
    elif periodontal:
        internal = "POSITIVE"
        tier = 1.00 if perio_from_code else 0.75
        source = "code:K05.3" if perio_from_code else "phrase:periodontitis"
    elif perio_endo:
        internal = "UNCERTAIN"
        tier = 0.25
        source = "perio_endo_ambiguous"
    elif other_perio:
        internal = "UNCERTAIN"
        tier = 0.25
        source = "other_periodontal_deferred"
    elif has_signal:
        internal = "NEGATIVE"
        tier = 1.00 if has_any_code else 0.75
        source = (
            "record_negative_code"
            if has_any_code
            else "record_negative_phrase"
        )
    else:
        internal = "UNCERTAIN"
        tier = 0.50
        source = "no_recorded_signal"

    if phrase_only_perio_endo:
        subgroup = "PERIO_ENDO_AMBIGUOUS"
    elif periodontal:
        subgroup = "PERIODONTITIS"
    elif perio_endo:
        subgroup = "PERIO_ENDO_AMBIGUOUS"
    elif other_perio:
        subgroup = "OTHER_PERIODONTAL"
    elif not has_signal:
        subgroup = "NO_RECORDED_SIGNAL"
    elif apical and gingivitis:
        subgroup = "APICAL_AND_GINGIVITIS"
    elif apical:
        subgroup = "APICAL_ONLY"
    elif gingivitis:
        subgroup = "GINGIVITIS_ONLY"
    elif pericoronal:
        subgroup = "PERICORONAL"
    else:
        subgroup = "OTHER_NONPERIODONTAL"

    if internal == "POSITIVE":
        ds_A, label_A = True, 1
    elif internal == "NEGATIVE":
        ds_A, label_A = True, 0
    else:
        ds_A, label_A = False, pd.NA

    if internal == "POSITIVE" or other_perio:
        ds_B, label_B = True, 1
    elif internal == "NEGATIVE":
        ds_B, label_B = True, 0
    else:
        ds_B, label_B = False, pd.NA

    if internal == "POSITIVE":
        label_C = "POSITIVE"
    elif internal == "NEGATIVE":
        label_C = "NEGATIVE"
    else:
        label_C = "UNLABELED"

    return {
        "internal_label": internal,
        "audit_subgroup": subgroup,
        "dataset_A_member": ds_A,
        "label_A_binary": label_A,
        "dataset_B_member": ds_B,
        "label_B_binary": label_B,
        "dataset_C_label": label_C,
        "binary_label_clean": label_A,
        "binary_label_expanded": label_B,
        "apical_aux_label": 1 if apical else 0,
        "reliability_tier": tier,
        "sample_weight": tier,
        "label_source": source,
    }


def build_label_table(records_df, cbct_ids=None):
    """One row per unique Filename with the full label/audit schema."""
    if ID_COL not in records_df.columns or DIAGNOSIS_COL not in records_df.columns:
        raise ValueError(f"records must contain '{ID_COL}' and '{DIAGNOSIS_COL}'")

    rows = []
    for fid, grp in records_df.groupby(ID_COL, sort=False):
        visit_texts = grp[DIAGNOSIS_COL].tolist()
        agg, has_sub, pcode, pphrase, codes, matched = aggregate_patient(visit_texts)
        has_signal = has_sub or len(codes) > 0
        labels = assign_labels(agg, has_signal, pcode, pphrase, len(codes) > 0)

        evidence_bits = sorted(k for k in matched if k != "nonperio_code")
        rec = {
            ID_COL: str(fid),
            "n_visits": len(grp),
            "n_diagnosed_visits": int(sum(1 for t in visit_texts
                                          if is_substantive(t) or extract_codes(t if isinstance(t, str) else ""))),
            "has_cbct": (str(fid) in cbct_ids) if cbct_ids is not None else pd.NA,
            "periodontal_present": int(agg["periodontal"]),
            "other_periodontal_present": int(agg["other_periodontal"]),
            "apical_present": int(agg["apical"]),
            "gingivitis_present": int(agg["gingivitis"]),
            "perio_endo_present": int(agg["perio_endo"]),
            "pericoronal_present": int(agg["pericoronal"]),
            "has_recorded_signal": int(has_signal),
            "codes_found": ";".join(sorted(set(codes))),
            "label_evidence": (labels["label_source"]
                               + (" | flags=" + ",".join(evidence_bits) if evidence_bits else "")),
        }
        rec.update(labels)
        rows.append(rec)

    out = pd.DataFrame(rows)
    if not out[ID_COL].is_unique:
        raise AssertionError("patient IDs are not unique after aggregation")
    return out


# --------------------------------------------------------------------------- #
# Regression tests (single-cell groups)                                        #
# --------------------------------------------------------------------------- #
def _label_of(diagnosis):
    df = pd.DataFrame({ID_COL: ["p"], DIAGNOSIS_COL: [diagnosis]})
    return build_label_table(df).iloc[0]


def run_regression_tests():
    cases = [
        # (diagnosis, periodontal, apical, gingivitis, other_perio, perio_endo, subgroup, clean)
        ("*31,*41 Chronic periodontitis (K05.300)", 1, 0, 0, 0, 0, "PERIODONTITIS", 1),
        ("k05.300 chronic periodontitis", 1, 0, 0, 0, 0, "PERIODONTITIS", 1),
        ("*46Chronic periodontitis", 1, 0, 0, 0, 0, "PERIODONTITIS", 1),                  # glued marker
        ("*46Chronic apical periodontitis (K04.500)", 0, 1, 0, 0, 0, "APICAL_ONLY", 0),  # glued + apical
        ("chronic apical periodontitis", 0, 1, 0, 0, 0, "APICAL_ONLY", 0),
        ("periradicular periodontitis", 0, 1, 0, 0, 0, "APICAL_ONLY", 0),
        ("*15 Chronic apical periodontitis (K04.500) *16 chronic periodontitis (K05.300)",
         1, 1, 0, 0, 0, "PERIODONTITIS", 1),                                              # both -> positive + aux
        ("Whole-mouth chronic gingivitis (K05.100)", 0, 0, 1, 0, 0, "GINGIVITIS_ONLY", 0),
        ("Pericoronal abscess (K05.202)", 0, 0, 0, 0, 0, "PERICORONAL", 0),
        ("acute pericoronitis (K05.204)", 0, 0, 0, 0, 0, "PERICORONAL", 0),
        ("Periodontal disease (K05.600)", 0, 0, 0, 1, 0, "OTHER_PERIODONTAL", None),       # excluded from A
        ("Periodontal abscess (K05.201)", 0, 0, 0, 1, 0, "OTHER_PERIODONTAL", None),
        ("periodontal pulp syndrome (K05.501)", 0, 0, 0, 0, 1, "PERIO_ENDO_AMBIGUOUS", None),
        ("bleeding at the gum margin, no periodontal pockets, calm", 0, 0, 0, 0, 0,
         "OTHER_NONPERIODONTAL", 0),                                                       # negated sign
        ("probing revealed no loose teeth and no periodontitis", 0, 0, 0, 0, 0,
         "OTHER_NONPERIODONTAL", 0),                                                       # negated periodontitis
        ("Caries (K02.900)", 0, 0, 0, 0, 0, "OTHER_NONPERIODONTAL", 0),
        ("Same as initial diagnosis", 0, 0, 0, 0, 0, "NO_RECORDED_SIGNAL", None),
        (None, 0, 0, 0, 0, 0, "NO_RECORDED_SIGNAL", None),
        (
            "Chronic periodontitis; combined periodontal and pulpal lesions",
            1,
            0,
            0,
            0,
            1,
            "PERIO_ENDO_AMBIGUOUS",
            None,
        ),
        (
            "No chronic periodontitis",
            0,
            0,
            0,
            0,
            0,
            "OTHER_NONPERIODONTAL",
            0,
        ),
        (
            "Chronic periodontitis with no pain",
            1,
            0,
            0,
            0,
            0,
            "PERIODONTITIS",
            1,
        ),
        (
            "K05.300 chronic periodontitis",
            1,
            0,
            0,
            0,
            0,
            "PERIODONTITIS",
            1,
        ),
    ]
    for (dx, perio, apic, ging, oth, pe, sub, clean) in cases:
        r = _label_of(dx)
        assert r["periodontal_present"] == perio, (dx, "periodontal", r["periodontal_present"])
        assert r["apical_present"] == apic, (dx, "apical", r["apical_present"])
        assert r["gingivitis_present"] == ging, (dx, "gingivitis", r["gingivitis_present"])
        assert r["other_periodontal_present"] == oth, (dx, "other_perio", r["other_periodontal_present"])
        assert r["perio_endo_present"] == pe, (dx, "perio_endo", r["perio_endo_present"])
        assert r["audit_subgroup"] == sub, (dx, "subgroup", r["audit_subgroup"])
        if clean is None:
            assert pd.isna(r["binary_label_clean"]), (dx, "clean NA", r["binary_label_clean"])
        else:
            assert r["binary_label_clean"] == clean, (dx, "clean", r["binary_label_clean"])
    print("[ok] label regression tests passed:", len(cases), "cases")

run_regression_tests()
print('label functions loaded and self-tested')


[ok] label regression tests passed: 22 cases
label functions loaded and self-tested


## 6. LABEL REGRESSION TESTS, COHORT SUMMARIES, AND AUDIT EXAMPLES


In [7]:
# Build the patient-level label table (has_cbct filled in Section 7).
patient_label_table = build_label_table(records_clean[['Filename', 'Diagnosis']])
N = len(patient_label_table)
print(f"patients labelled: {N}")
assert patient_label_table['Filename'].is_unique, "duplicate Filename in label table"
assert N == records_clean['Filename'].astype(str).nunique(), "row count != unique patients"

def show_counts(df, scope):
    n = len(df)
    print(f"\n=== {scope} | n={n} ===")
    print("audit_subgroup:"); print(df['audit_subgroup'].value_counts().to_string())
    print("\nreliability_tier:"); print(df['reliability_tier'].value_counts().sort_index().to_string())
    a = df[df['dataset_A_member']]; b = df[df['dataset_B_member']]
    pa, na = int((a['label_A_binary']==1).sum()), int((a['label_A_binary']==0).sum())
    pb, nb = int((b['label_B_binary']==1).sum()), int((b['label_B_binary']==0).sum())
    print(f"\nDataset A: {pa} pos / {na} neg / {n-len(a)} excluded | prevalence {pa/max(pa+na,1):.1%}")
    print(f"Dataset B: {pb} pos / {nb} neg / {n-len(b)} excluded | prevalence {pb/max(pb+nb,1):.1%}")
    c = df['dataset_C_label'].value_counts()
    print(f"Dataset C: POS {int(c.get('POSITIVE',0))} / NEG {int(c.get('NEGATIVE',0))} / UNLABELED {int(c.get('UNLABELED',0))}")

show_counts(patient_label_table, "ALL RECORD PATIENTS")

# ---- hard label invariants ----
t = patient_label_table
apical_only_labels = pd.to_numeric(
    t.loc[
        t["audit_subgroup"] == "APICAL_ONLY",
        "label_A_binary",
    ],
    errors="coerce",
)

assert (
    apical_only_labels.fillna(-1) != 1
).all(), "apical-only must not be strict positive"

# pure ambiguous (perio-endo with no periodontal and no other-periodontal evidence) -> excluded from A and B
_pure_pe = ((t['perio_endo_present']==1) & (t['periodontal_present']==0) & (t['other_periodontal_present']==0))
assert (t[_pure_pe]['dataset_A_member'] == False).all(), "pure perio-endo must be excluded from A"  # noqa: E712
assert (t[_pure_pe]['dataset_B_member'] == False).all(), "pure perio-endo must be excluded from B"  # noqa: E712
assert (t[t['audit_subgroup']=='OTHER_PERIODONTAL']['dataset_A_member'] == False).all(), "other-periodontal excluded from A"  # noqa: E712
assert (t[t['audit_subgroup']=='OTHER_PERIODONTAL']['label_B_binary'] == 1).all(), "other-periodontal is B-positive"
assert (t[(t['has_recorded_signal']==0)]['dataset_C_label'] == 'UNLABELED').all(), "no-signal must be UNLABELED"
# code-level guarantees, verified directly on the extractor
assert _label_of("Pericoronal abscess (K05.202)")['periodontal_present'] == 0
assert _label_of("acute pericoronitis (K05.204)")['periodontal_present'] == 0
assert pd.isna(_label_of("Periodontal abscess (K05.201)")['binary_label_clean'])      # B only, not strict A
assert _label_of("Periodontal abscess (K05.201)")['binary_label_expanded'] == 1
assert _label_of("*31 Chronic periodontitis (K05.300)")['binary_label_clean'] == 1
assert _label_of("Whole-mouth chronic gingivitis (K05.100)")['periodontal_present'] == 0
assert _label_of("chronic apical periodontitis (K04.500)")['periodontal_present'] == 0
assert _label_of("no periodontal pocket, no loose teeth")['periodontal_present'] == 0
print("\n[ok] all label invariants hold")

# ---- back-compat alias with explicit warning ----
patient_label_table['binary_label'] = patient_label_table['binary_label_clean']
log.warning("`binary_label` is now an ALIAS for `binary_label_clean` (strict Dataset A). "
            "Downstream code must default to `binary_label_clean`.")

# ---- representative auditable examples (computed, not hardcoded) ----
print("\n--- representative examples ---")
for sg in ['PERIODONTITIS','APICAL_ONLY','APICAL_AND_GINGIVITIS','GINGIVITIS_ONLY',
           'PERICORONAL','PERIO_ENDO_AMBIGUOUS','OTHER_NONPERIODONTAL','NO_RECORDED_SIGNAL']:
    r = patient_label_table[patient_label_table['audit_subgroup']==sg].head(1)
    if len(r):
        r = r.iloc[0]
        print(f"[{sg}] Filename={r['Filename']} clean={r['binary_label_clean']} "
              f"aux={r['apical_aux_label']} tier={r['reliability_tier']} "
              f"codes={r['codes_found'][:50]!r} :: {r['label_evidence'][:70]}")


patients labelled: 660

=== ALL RECORD PATIENTS | n=660 ===
audit_subgroup:
audit_subgroup
OTHER_NONPERIODONTAL     363
GINGIVITIS_ONLY           81
APICAL_ONLY               61
PERIODONTITIS             59
OTHER_PERIODONTAL         40
NO_RECORDED_SIGNAL        21
PERICORONAL               17
APICAL_AND_GINGIVITIS     11
PERIO_ENDO_AMBIGUOUS       7

reliability_tier:
reliability_tier
0.25     47
0.50     21
0.75     84
1.00    508

Dataset A: 59 pos / 533 neg / 68 excluded | prevalence 10.0%
Dataset B: 100 pos / 533 neg / 27 excluded | prevalence 15.8%
Dataset C: POS 59 / NEG 533 / UNLABELED 68

[ok] all label invariants hold

--- representative examples ---
[PERIODONTITIS] Filename=8 clean=1 aux=0 tier=1.0 codes='K05.300' :: code:K05.3 | flags=periodontal
[APICAL_ONLY] Filename=4 clean=0 aux=1 tier=0.75 codes='' :: record_negative_phrase | flags=apical
[APICAL_AND_GINGIVITIS] Filename=134 clean=0 aux=1 tier=1.0 codes='K04.500;K05.100;K08.302' :: record_negative_code | flags=apical,gi

## 6A. GENERATE REVIEWED STAGE-II SILVER LABELS

In [8]:
# ============================================================================
# Section 6a: generate the reviewed Stage-II SILVER labels in-memory.
# Merges the two adjudication passes so the notebook is self-contained and does
# NOT require a pre-existing mmd_patient_labels_refined_reviewed.csv:
#   medical_records.csv
#     -> build_stage1_refined()  (first-pass refined labels, kept in memory)
#     -> build_stage2_reviewed() (second-pass reviewed/adjudicated labels)
#     -> save ONLY mmd_patient_labels_refined_reviewed.csv (no intermediate CSVs)
#
# These are REVIEWED RECORD-DERIVED SILVER labels, not expert-confirmed strong/
# gold labels. Expert-confirmed gold labels still require expert CBCT review.
# ============================================================================

def build_stage1_refined(raw_df):
    """First-pass refined labels (from label_engine.py), returned in memory."""
    TEXT_COLS = ['Main appeal', 'Subsequent', 'Present medical history',
                 'Past medical history', 'Oral Check', 'Diagnosis',
                 'Treatment plan', 'Handle', 'Doctor advices']

    df = raw_df.copy()

    # ---------------------------------------------------------------------------
    # Helpers
    # ---------------------------------------------------------------------------
    def norm(s: str) -> str:
        s = str(s) if pd.notna(s) else ''
        s = s.replace('&#39;', "'").replace('&quot;', '"').replace('&amp;', '&')
        return s

    def has(pattern, text, flags=re.I):
        return re.search(pattern, text, flags) is not None

    def strip_negated_pockets(text):
        """Remove negated pocket spans so they don't trigger positive pocket evidence."""
        neg = [
            r'no periodontal pocket[s]?',
            r'periodontal pocket[s]? (?:are|is|were)? ?not reach(?:ed)?',
            r'pocket[s]? (?:are|is|were)? ?not reach(?:ed)?',
            r'no (?:obvious )?(?:deep )?pocket[s]?',
            r'without periodontal pocket[s]?',
        ]
        for n in neg:
            text = re.sub(n, ' [NEG_POCKET] ', text, flags=re.I)
        return text

    # short evidence snippet extractor
    def snippet(text, pattern, width=70):
        m = re.search(pattern, text, re.I)
        if not m:
            return ''
        a = max(0, m.start() - width); b = min(len(text), m.end() + width)
        return re.sub(r'\s+', ' ', text[a:b]).strip()

    # ---------------------------------------------------------------------------
    # Per-patient aggregation
    # ---------------------------------------------------------------------------
    records = []
    for pid, g in df.groupby('Filename'):
        n_visits = len(g)
        # per-visit concatenated text (for conflict detection) + global blob
        visit_blobs = []
        all_codes = []
        diag_present_any = False
        for _, row in g.iterrows():
            parts = [norm(row[c]) for c in TEXT_COLS]
            vb = ' || '.join(p for p in parts if p)
            visit_blobs.append(vb)
            if pd.notna(row['Diagnosis']) and str(row['Diagnosis']).strip():
                diag_present_any = True
            all_codes += re.findall(r'[A-Z]\d{2}\.\d{3}(?:x\d{3})?|[A-Z]\d{2}\.\d{2,3}|[A-Z]\d{2}\.\d', norm(row['Diagnosis']))
        blob_raw = ' '.join(visit_blobs)
        blob = strip_negated_pockets(blob_raw)
        low = blob.lower()
        codes = sorted(set(all_codes))

        # ---- code-based signals ----
        c_chronic_perio = has(r'K05\.3', blob)                       # chronic periodontitis
        c_periodontosis = has(r'K05\.4', blob)                       # periodontal degeneration
        c_perio_abscess = has(r'K05\.201', blob)                     # periodontal abscess
        c_pericoronal   = has(r'K05\.202', blob) or has(r'pericoron', blob)  # trap
        c_periopulp     = has(r'K05\.501', blob)                     # perio-pulp syndrome
        c_perio_unspec  = has(r'K05\.6', blob)                       # unspecified perio disease
        c_gingivitis    = has(r'K05\.[01]', blob) or has(r'gingivitis', blob)
        c_apical        = has(r'K04\.5', blob) or has(r'apical periodont', blob)

        # ---- text-based periodontal signals (exclude apical) ----
        # explicit periodontitis word NOT immediately preceded by "apical"
        txt_periodontitis = bool(re.search(r'(?<!apical )(?<!apical)\bperiodontitis\b', blob, re.I)) \
                            and not (has(r'apical periodont', blob) and not c_chronic_perio
                                     and not c_perio_unspec and not has(r'chronic periodontitis', blob))
        # be strict: require a non-apical periodontitis mention
        txt_periodontitis = has(r'chronic periodontitis|aggressive periodontitis|whole-?mouth .{0,20}periodontitis|marginal periodontitis|periodontitis \(K05', blob)

        # periodontal treatment evidence
        perio_tx = has(r'root planing|subgingival (?:scaling|curettage)|periodontal flap|guided tissue|periodontal (?:treatment|therapy|maintenance)|deep (?:scaling|cleaning)', blob)

        # periodontal bone loss (alveolar / horizontal-vertical), not periapical shadow
        perio_bone = has(r'alveolar bone (?:resorption|loss|atrophy)|horizontal bone (?:loss|resorption)|vertical bone (?:loss|defect)|resorption of (?:the )?alveolar', blob)

        # positive pocket evidence (negated spans already stripped)
        pocket_pos = has(r'pocket depth|periodontal pocket (?:of|formation|reach|=|:)|deep periodontal pocket|pocket[s]? (?:of )?\d\s?mm|PD[=: ]?\d', blob)
        pocket_neg = has(r'\[NEG_POCKET\]', blob)

        tooth_mobility = has(r'\bmobility\b|loosening of teeth|tooth loosening|degree (?:I|II|III) loosening|loose(?:ness)? of teeth', blob)

        # ---- traps ----
        t_impacted     = has(r'impacted|embedded tooth|K01\.1', blob)
        t_extraction   = has(r'extraction|extracted|K08\.1|missing teeth|tooth loss|K00\.001', blob)
        t_ortho        = has(r'orthodontic|malocclusion|K07\.3|correction|brackets', blob)
        t_cbct_monitor = has(r'cbct .{0,30}(?:monitor|regularly|follow)|monitor .{0,20}cbct|take cbct', blob)
        t_fenestration = has(r'fenestration|bone crack|dehiscence', blob)
        t_thin_alv     = has(r'thin .{0,15}alveolar|alveolar bone is thin|thin bone', blob)
        t_residual     = has(r'residual root|K08\.3', blob)
        t_gingiv_only  = c_gingivitis and not (c_chronic_perio or c_periodontosis or c_perio_abscess
                                               or c_perio_unspec or txt_periodontitis or perio_bone or pocket_pos)
        t_apical_only  = c_apical and not (c_chronic_perio or c_periodontosis or c_perio_abscess
                                           or c_perio_unspec or c_periopulp or txt_periodontitis
                                           or perio_bone or pocket_pos)
        t_pericor_only = c_pericoronal and not (c_chronic_perio or c_periodontosis or c_perio_abscess
                                                or c_perio_unspec or txt_periodontitis or perio_bone or pocket_pos)

        missing_diag = not diag_present_any
        thin_oral_check = (g['Oral Check'].fillna('').str.len().max() < 5)

        # explicit healthy-periodontium negatives
        healthy_perio = pocket_neg or has(r'no (?:obvious )?loosening|gums are (?:healthy|normal|good)|no periodontal (?:disease|problem)|gingiva.{0,15}(?:healthy|normal)', blob)

        # ---- strong periodontitis positive signal ----
        strong_pos = c_chronic_perio or c_periodontosis or c_perio_abscess or \
                     (txt_periodontitis and (perio_tx or pocket_pos or perio_bone))

        # any real periodontal signal
        any_perio_pos = strong_pos or c_perio_unspec or c_periopulp or txt_periodontitis or \
                        perio_bone or pocket_pos or perio_tx

        # ---- conflict detection across visits ----
        pos_visits = sum(1 for vb in visit_blobs
                         if has(r'K05\.3|K05\.4|K05\.6|K05\.201|chronic periodontitis', vb))
        neg_visit_signal = healthy_perio
        conflicting = strong_pos and neg_visit_signal

        # =========================================================================
        # LABEL STATE DECISION (priority ordered)
        # =========================================================================
        reasons = []
        codes_found = ', '.join(codes) if codes else ''

        if strong_pos:
            state = 'confirmed_positive'
            rlabel = 1
            conf = 0.95
            if conflicting:
                conf = 0.78
                reasons.append('positive code present but a visit records healthy/no-pocket periodontium (conflicting visits)')
            if pos_visits == 1 and n_visits > 3 and not (perio_tx or perio_bone or pocket_pos):
                conf = min(conf, 0.85)
                reasons.append('single-visit code mention among many visits')
            src = 'ICD_code' if (c_chronic_perio or c_periodontosis or c_perio_abscess) else 'clinical_text+treatment'

        elif c_perio_unspec or c_periopulp or (txt_periodontitis and not strong_pos):
            state = 'probable_positive'
            rlabel = 1
            conf = 0.68
            src = 'ICD_code' if (c_perio_unspec or c_periopulp) else 'clinical_text'
            reasons.append('periodontal disease indicated but unstaged/uncorroborated; CBCT can confirm bone loss')

        elif perio_bone or pocket_pos or (perio_tx and not t_gingiv_only):
            state = 'probable_positive'
            rlabel = 1
            conf = 0.62
            src = 'clinical_findings'
            reasons.append('indirect periodontal findings (bone loss / pocketing / perio treatment) without explicit diagnosis')

        elif t_gingiv_only:
            state = 'review_required'
            rlabel = -1
            conf = 0.40
            src = 'gingivitis_only'
            reasons.append('gingivitis-only record; may be reversible or mask early bone loss — CBCT review needed')

        elif t_apical_only:
            state = 'probable_negative'
            rlabel = 0
            conf = 0.60
            src = 'apical_disease'
            reasons.append('apical (periapical/pulpal) periodontitis only — distinct from marginal periodontitis; not a periodontal positive')

        elif t_pericor_only:
            state = 'review_required'
            rlabel = -1
            conf = 0.42
            src = 'pericoronal_disease'
            reasons.append('pericoronal abscess/pericoronitis around impacted tooth — localized, not generalized periodontitis')

        elif missing_diag or thin_oral_check:
            state = 'uncertain'
            rlabel = -1
            conf = 0.28
            src = 'insufficient_record'
            reasons.append('missing diagnosis / sparse oral-check text — cannot label without CBCT + chart review')

        elif diag_present_any and not any_perio_pos and healthy_perio and \
             not (t_thin_alv or t_fenestration or t_cbct_monitor):
            state = 'confirmed_negative'
            rlabel = 0
            conf = 0.88
            src = 'explicit_negative_findings'
            reasons.append('non-periodontal diagnosis with explicit healthy periodontium (no pockets / no mobility)')

        elif diag_present_any and not any_perio_pos:
            state = 'probable_negative'
            rlabel = 0
            conf = 0.62
            src = 'non_periodontal_diagnosis'
            reasons.append('non-periodontal diagnosis, no periodontal evidence, but periodontium not explicitly assessed as normal')

        else:
            state = 'uncertain'
            rlabel = -1
            conf = 0.30
            src = 'ambiguous'
            reasons.append('ambiguous / borderline evidence')

        # ---- trap overlays: bump toward expert review even when otherwise labeled ----
        trap_notes = []
        if t_thin_alv:      trap_notes.append('thin alveolar bone')
        if t_fenestration:  trap_notes.append('bone fenestration/cracking risk')
        if t_cbct_monitor:  trap_notes.append('CBCT monitoring context')
        if t_ortho:         trap_notes.append('orthodontic treatment')
        if t_extraction:    trap_notes.append('tooth extraction / missing teeth history')
        if t_impacted:      trap_notes.append('impacted tooth')
        if t_residual:      trap_notes.append('residual root')
        if t_apical_only:   trap_notes.append('apical disease')
        if t_gingiv_only:   trap_notes.append('gingivitis-only')
        if t_pericor_only:  trap_notes.append('pericoronal disease')
        if conflicting:     trap_notes.append('conflicting visits')
        if missing_diag:    trap_notes.append('missing diagnosis')

        # confirmed_negative with a bone-hiding trap must not stay "confirmed"
        if state == 'confirmed_negative' and (t_thin_alv or t_fenestration or t_cbct_monitor):
            state = 'review_required'; conf = min(conf, 0.5)
            reasons.append('downgraded: thin bone / fenestration / CBCT-monitoring context can hide or mimic bone loss')

        # A probable_negative is a usable SOFT negative unless a bone-relevant or
        # explicitly-listed confound is present (apical/gingivitis/pericoronal disease,
        # thin bone, fenestration risk, CBCT-monitoring context, or conflict).
        bone_relevant_trap = (t_thin_alv or t_fenestration or t_cbct_monitor
                              or t_apical_only or t_gingiv_only or t_pericor_only or conflicting)
        if state in ('probable_positive', 'review_required', 'uncertain'):
            expert = True
        elif state == 'confirmed_positive':
            expert = conflicting
        elif state == 'confirmed_negative':
            expert = False
        elif state == 'probable_negative':
            expert = bool(bone_relevant_trap)
        else:
            expert = True

        # sample weight: full for confident hard labels, tapered otherwise
        weight_map = {
            'confirmed_positive': 1.0, 'confirmed_negative': 1.0,
            'probable_positive': 0.6, 'probable_negative': 0.5,
            'review_required': 0.25, 'uncertain': 0.1,
        }
        weight = weight_map[state]
        if conflicting:
            weight = min(weight, 0.4)

        # review priority: value of expert CBCT confirmation
        if state == 'confirmed_positive' and not conflicting:
            prio = 'low'
        elif state == 'confirmed_negative':
            prio = 'low'
        elif state in ('probable_positive',) or conflicting:
            prio = 'high'
        elif state == 'review_required':
            prio = 'high' if (perio_bone or pocket_pos or t_gingiv_only) else 'medium'
        elif state == 'uncertain':
            prio = 'medium'
        else:
            prio = 'medium'

        # ---- original WEAK label (documented naive rule) ----
        # naive keyword rule: positive if the word "periodontitis" or a K05.3 code
        # appears anywhere in the patient's records (conflates apical & gingivitis).
        original_binary = int(has(r'periodontitis|K05\.3', blob_raw))

        # ---- evidence snippet ----
        ev_pat = (r'chronic periodontitis|K05\.3|K05\.6|K05\.201|periodontal pocket|'
                  r'alveolar bone (?:resorption|loss)|apical periodont|gingivitis|pericoron')
        ev = snippet(blob_raw, ev_pat) or snippet(blob_raw, r'K0\d\.\d') or (blob_raw[:120])
        ev = re.sub(r'\s+', ' ', ev)[:240]

        records.append({
            'patient_id': int(pid),
            'original_binary_label': original_binary,
            'refined_periodontitis_label': rlabel,       # 1 pos / 0 neg / -1 abstain
            'label_state': state,
            'label_confidence': round(conf, 2),
            'label_source': src,
            'evidence_text': ev,
            'codes_found': codes_found,
            'review_priority': prio,
            'review_reason': '; '.join(reasons) + (('; traps: ' + ', '.join(trap_notes)) if trap_notes else ''),
            'recommended_sample_weight': round(weight, 2),
            'expert_review_required': bool(expert),
            'n_visits': n_visits,
        })

    out = pd.DataFrame(records).sort_values('patient_id').reset_index(drop=True)
    return out


def build_stage2_reviewed(raw_df, stage1_df):
    """Second-pass reviewed/adjudicated labels (from stage2_review.py), in memory."""
    TEXT_COLS = ['Main appeal','Subsequent','Present medical history','Past medical history',
                 'Oral Check','Diagnosis','Treatment plan','Handle','Doctor advices']

    raw = raw_df.copy()
    s1 = stage1_df.copy()

    def norm(s):
        s = str(s) if pd.notna(s) else ''
        return s.replace('&#39;',"'").replace('&quot;','"').replace('&amp;','&')
    def has(p,t,f=re.I): return re.search(p,t,f) is not None
    def strip_neg_pockets(t):
        for n in [r'no periodontal pocket[s]?', r'periodontal pocket[s]? (?:are|is|were)? ?not reach(?:ed)?',
                  r'pocket[s]? (?:are|is|were)? ?not reach(?:ed)?', r'no (?:obvious )?(?:deep )?pocket[s]?',
                  r'without periodontal pocket[s]?', r'does not reach (?:deep )?periodontal pocket[s]?']:
            t = re.sub(n,' [NEG_POCKET] ',t,flags=re.I)
        return t
    def snippet(t,p,w=80):
        m=re.search(p,t,re.I)
        if not m: return ''
        a=max(0,m.start()-w); b=min(len(t),m.end()+w)
        return re.sub(r'\s+',' ',t[a:b]).strip()

    # ---- per-patient signal extraction (flagged cases re-read from raw) ----
    def signals(g):
        visit_blobs=[]; codes=[]; diag_any=False
        for _,row in g.iterrows():
            parts=[norm(row[c]) for c in TEXT_COLS]
            visit_blobs.append(' || '.join(p for p in parts if p))
            if pd.notna(row['Diagnosis']) and str(row['Diagnosis']).strip(): diag_any=True
            codes += re.findall(r'[A-Z]\d{2}\.\d{3}(?:x\d{3})?|[A-Z]\d{2}\.\d{2,3}|[A-Z]\d{2}\.\d', norm(row['Diagnosis']))
        raw_blob=' '.join(visit_blobs); blob=strip_neg_pockets(raw_blob)
        d=dict(
            chronic_perio = has(r'K05\.3',blob),
            periodontosis = has(r'K05\.4',blob),
            perio_abscess = has(r'K05\.201',blob),
            pericoronal   = has(r'K05\.202',blob) or has(r'pericoron',blob),
            periopulp     = has(r'K05\.501',blob),
            perio_unspec  = has(r'K05\.6',blob),
            gingivitis    = has(r'K05\.[01]',blob) or has(r'gingivitis',blob),
            apical        = has(r'K04\.5',blob) or has(r'apical periodont',blob),
            txt_perio     = has(r'chronic periodontitis|aggressive periodontitis|whole-?mouth .{0,20}periodontitis|marginal periodontitis|periodontitis \(K05',blob),
            perio_tx      = has(r'root planing|subgingival (?:scaling|curettage)|periodontal flap|guided tissue|periodontal (?:treatment|therapy|maintenance)|deep (?:scaling|cleaning)',blob),
            perio_bone    = has(r'alveolar bone (?:resorption|loss|atrophy)|horizontal bone (?:loss|resorption)|vertical bone (?:loss|defect)|resorption of (?:the )?alveolar|bone (?:has been )?absorbed to the (?:apical|middle|cervical)',blob),
            pocket_pos    = has(r'pocket depth|periodontal pocket (?:of|formation|reach|=|:)|deep periodontal pocket|pocket[s]? (?:of )?\d\s?mm|PD[=: ]?\d',blob),
            pocket_neg    = has(r'\[NEG_POCKET\]',blob),
            mobility      = has(r'\bmobility\b|loosening of teeth|tooth loosening|degree (?:I|II|III) loosening|loose(?:ness)? of teeth',blob),
            healthy_gums  = has(r'gums? (?:are |is )?(?:healthy|normal|good)|gingiva.{0,15}(?:healthy|normal)|gums? (?:are |is )?not red|no red(?:ness)?(?: and| or)? swell|no swelling (?:and |or )?red|gums? (?:are )?not (?:red|swollen)|no (?:obvious )?(?:redness|abnormal)|no redness and swelling',blob),
            # dataset convention: "(-)" = sign absent, "( )"/"(+)" = sign present
            mobility_pos  = has(r'(?:loosen(?:ing|ess)?|loose|mobility)\s*\(\s*\+?\s*\)|grade\s*(?:i{1,3}|[123])\b.{0,14}(?:loose|looseness|mobility)|(?:i{1,3}|[123])\s*degree.{0,10}(?:loose|mobility)|(?:loose|looseness)\s*grade\s*(?:i{1,3}|[123])',blob),
            no_mobility   = has(r'(?:loosen(?:ing|ess)?|loose|mobility)\s*\(-\)|no (?:obvious )?loosening|teeth are not loose|not loose|no.{0,3}mobility',blob),
            thin_alv      = has(r'thin .{0,15}alveolar|alveolar bone is thin|thin bone|bone height (?:is|was) insufficient|insufficient bone',blob),
            fenestration  = has(r'fenestration|bone crack|dehiscence',blob),
            cbct_monitor  = has(r'cbct .{0,30}(?:monitor|regularly|follow)|monitor .{0,20}cbct|take cbct .{0,20}regular',blob),
            active_ortho  = has(r'brackets|during the process|during correction|orthodontic .{0,20}(?:process|ongoing)|retract .{0,10}(?:teeth|anterior)|close the (?:maxillary )?gap',blob),
            missing_diag  = (not diag_any),
            residual      = has(r'residual root|K08\.3',blob),
            impacted      = has(r'impacted|embedded tooth|K01\.1',blob),
            extraction    = has(r'extraction|extracted|missing teeth|tooth loss|K00\.001|K08\.1',blob),
        )
        d['raw_blob']=raw_blob; d['blob']=blob; d['codes']=sorted(set(codes)); d['diag_any']=diag_any
        # apical-only / gingivitis-only / pericoronal-only helpers
        any_true_perio = d['chronic_perio'] or d['periodontosis'] or d['perio_abscess'] or d['perio_unspec'] or d['txt_perio'] or d['perio_bone'] or d['pocket_pos']
        d['apical_only']    = d['apical'] and not any_true_perio and not d['periopulp']
        d['gingiv_only']    = d['gingivitis'] and not any_true_perio
        d['pericor_only']   = d['pericoronal'] and not any_true_perio
        d['explicit_normal_perio'] = (not d['mobility_pos']) and (
            d['pocket_neg'] or (d['healthy_gums'] and d['no_mobility']))
        return d

    # ---- resolution decision for a flagged patient ----
    def resolve(d):
        hard_keep = (d['gingiv_only'] or d['pericor_only'] or d['thin_alv'] or d['fenestration']
                     or d['cbct_monitor'] or d['active_ortho'] or d['missing_diag'])

        strong_pos = d['chronic_perio'] or d['periodontosis'] or d['perio_abscess'] or d['txt_perio']
        corroborated = d['perio_bone'] or (d['pocket_pos'] and not d['pocket_neg']) or d['perio_tx']

        # ---------- POSITIVE ----------
        if strong_pos and not (d['apical_only']):
            conf = 0.88 if (d['perio_bone'] or corroborated) else 0.82
            wt   = 0.88 if (d['perio_bone'] or corroborated) else 0.78
            why = 'clinician-assigned periodontitis (K05.3/K05.4 or explicit periodontitis text)'
            if d['healthy_gums'] or d['pocket_neg']:
                conf, wt = 0.80, 0.75
                why += '; note: another visit records a healthy periodontium (localized/partly resolved disease)'
            return ('record_resolved_positive', 1, conf, wt, why)

        if (d['perio_unspec'] or d['txt_perio'] or d['periopulp']) and corroborated:
            return ('record_resolved_positive', 1, 0.80, 0.75,
                    'periodontal diagnosis (K05.6/K05.501/text) corroborated by bone loss, pocketing, or periodontal treatment')

        if d['perio_bone'] and not d['apical_only']:
            return ('record_resolved_positive', 1, 0.78, 0.72,
                    'clear periodontal alveolar bone resorption/loss documented')

        # ---------- HARD-KEEP -> review ----------
        if hard_keep:
            reasons=[]
            if d['gingiv_only']:   reasons.append('gingivitis-only (CBCT needed to distinguish early periodontitis)')
            if d['pericor_only']:  reasons.append('pericoronal disease around impacted tooth')
            if d['thin_alv']:      reasons.append('thin alveolar bone / insufficient bone height')
            if d['fenestration']:  reasons.append('bone fenestration/dehiscence risk')
            if d['cbct_monitor']:  reasons.append('CBCT-monitoring context')
            if d['active_ortho']:  reasons.append('active orthodontic tooth movement')
            if d['missing_diag']:  reasons.append('missing diagnosis / insufficient record')
            conf = 0.35 if d['missing_diag'] else 0.45
            return ('expert_review_required', -1, conf, 0.0, 'hard-keep: ' + '; '.join(reasons))

        # unstaged periodontal disease without corroboration -> keep
        if d['perio_unspec'] or d['periopulp']:
            return ('expert_review_required', -1, 0.48, 0.0,
                    'unspecified/associated periodontal code (K05.6/K05.501) without corroborating bone loss, pocketing, or treatment — stage on CBCT')

        # tooth mobility is a periodontal attachment-loss red flag -> keep for CBCT
        if d['mobility_pos'] and not strong_pos:
            return ('expert_review_required', -1, 0.45, 0.0,
                    'tooth mobility/loosening documented (possible periodontal attachment loss) without a periodontitis diagnosis — confirm bone level on CBCT')

        # ---------- NEGATIVE ----------
        if d['diag_any'] and not (strong_pos or d['perio_unspec'] or d['periopulp'] or d['perio_bone']
                                  or (d['pocket_pos'] and not d['pocket_neg']) or d['perio_tx']):
            if d['apical_only']:
                if d['explicit_normal_perio']:
                    return ('record_resolved_negative', 0, 0.70, 0.60,
                            'apical (periapical) periodontitis only, with explicitly healthy marginal periodontium (no pockets / no mobility); periapical bone change is a CBCT caveat, not marginal periodontitis')
                return ('expert_review_required', -1, 0.48, 0.0,
                        'apical periodontitis with marginal periodontium not documented — periapical bone change may confound CBCT bone-level assessment')
            if d['explicit_normal_perio']:
                conf = 0.82 if (d['pocket_neg'] and d['no_mobility']) else 0.72
                wt   = 0.75 if (d['pocket_neg'] and d['no_mobility']) else 0.60
                return ('record_resolved_negative', 0, conf, wt,
                        'non-periodontal diagnosis with explicitly healthy periodontium (pockets not reached / no mobility) and no periodontal signal')
            return ('expert_review_required', -1, 0.50, 0.0,
                    'non-periodontal diagnosis but periodontium not explicitly assessed as normal — cannot confirm absence of bone loss without CBCT')

        return ('expert_review_required', -1, 0.40, 0.0, 'ambiguous / conflicting / insufficient evidence')

    # ---------------------------------------------------------------------------
    raw_by_pid = {pid:g for pid,g in raw.groupby('Filename')}
    rows=[]
    for _,r in s1.iterrows():
        pid=int(r['patient_id'])
        rec=dict(r)  # preserve all stage-1 columns
        rec['previous_label_state']=r['label_state']
        rec['previous_expert_review_required']=bool(r['expert_review_required'])
        if not bool(r['expert_review_required']):
            # carried through from stage-1 (already resolved)
            rec['review_resolution']='stage1_resolved'
            rec['resolved_periodontitis_label']=int(r['refined_periodontitis_label'])
            rec['resolved_label_state']=r['label_state']
            rec['resolved_confidence']=round(float(r['label_confidence']),2)
            rec['resolved_sample_weight']=round(float(r['recommended_sample_weight']),2)
            rec['expert_review_required_final']=False
            rec['resolution_reason']='not flagged in stage 1; label carried forward'
        else:
            d=signals(raw_by_pid[pid])
            resn, lab, conf, wt, why = resolve(d)
            rec['review_resolution']=resn
            rec['resolved_periodontitis_label']=lab
            rec['resolved_label_state']=resn
            rec['resolved_confidence']=round(conf,2)
            rec['resolved_sample_weight']=round(wt,2)
            rec['expert_review_required_final']=(resn=='expert_review_required')
            rec['resolution_reason']=why
            # refresh evidence + codes from a full re-read
            ev = snippet(d['raw_blob'], r'chronic periodontitis|K05\.3|K05\.6|K05\.201|K05\.501|periodontal pocket|alveolar bone (?:resorption|loss|absorbed)|apical periodont|gingivitis|pericoron') \
                 or snippet(d['raw_blob'], r'K0\d\.\d') or d['raw_blob'][:140]
            rec['evidence_text']=re.sub(r'\s+',' ',ev)[:260]
            rec['codes_found']=', '.join(d['codes'])
        rows.append(rec)

    out=pd.DataFrame(rows)
    col_order=['patient_id','original_binary_label','refined_periodontitis_label','label_state',
               'label_confidence','label_source','review_priority','n_visits',
               'previous_label_state','previous_expert_review_required',
               'review_resolution','resolved_periodontitis_label','resolved_label_state',
               'resolved_confidence','resolved_sample_weight','expert_review_required_final',
               'resolution_reason','evidence_text','codes_found']
    out=out[[c for c in col_order if c in out.columns]]
    return out


In [9]:
# ---- run both passes in memory, then save ONLY the final reviewed CSV ----
# Read the raw clinical records (the two passes do their own text normalisation);
# reading RECORDS_CSV reproduces the adjudication exactly.
raw_records = pd.read_csv(RECORDS_CSV, encoding='utf-8-sig')

refined_df = build_stage1_refined(raw_records)          # first-pass, in memory only
reviewed_df = build_stage2_reviewed(raw_records, refined_df)  # second-pass, in memory

# The only CSV this section writes (downstream Stage-II contract input).
MMD_REVIEWED_LABELS_CSV = MMD_LOG_DIR / "mmd_patient_labels_refined_reviewed.csv"
reviewed_df.to_csv(MMD_REVIEWED_LABELS_CSV, index=False, encoding='utf-8-sig')
os.environ["MMD_REVIEWED_LABELS_CSV"] = str(MMD_REVIEWED_LABELS_CSV)

# --------------------------- validation assertions ---------------------------
REVIEWED_REQUIRED_COLS = [
    'patient_id', 'resolved_periodontitis_label', 'resolved_sample_weight',
    'expert_review_required_final', 'review_resolution', 'resolved_label_state',
    'resolved_confidence', 'resolution_reason', 'evidence_text', 'codes_found',
    'original_binary_label', 'refined_periodontitis_label', 'label_state',
]
_missing = [c for c in REVIEWED_REQUIRED_COLS if c not in reviewed_df.columns]
assert not _missing, f"reviewed labels missing required columns: {_missing}"

assert reviewed_df['patient_id'].is_unique, "patient_id must be unique"

reviewed_df['resolved_periodontitis_label'] = reviewed_df['resolved_periodontitis_label'].astype(int)
assert reviewed_df['resolved_periodontitis_label'].isin([-1, 0, 1]).all(), \
    "resolved_periodontitis_label must be one of -1, 0, 1"

# expert_review_required_final must be boolean-like
_erf = reviewed_df['expert_review_required_final']
assert _erf.dtype == bool or _erf.map(lambda v: str(v).strip().lower()
                                      in ('true', 'false', '0', '1', '0.0', '1.0')).all(), \
    "expert_review_required_final must be boolean-like"
_erf_bool = _erf if _erf.dtype == bool else _erf.map(lambda v: str(v).strip().lower() in ('true', '1', '1.0'))

# unresolved (-1) rows: weight 0.0 and expert-review required
_unres = reviewed_df['resolved_periodontitis_label'] == -1
assert (reviewed_df.loc[_unres, 'resolved_sample_weight'].astype(float) == 0.0).all(), \
    "every unresolved (-1) row must have resolved_sample_weight == 0.0"
assert _erf_bool[_unres].all(), "every unresolved (-1) row must have expert_review_required_final == True"

# supervised (0/1) rows: weight > 0 and NOT expert-review required
_sup = reviewed_df['resolved_periodontitis_label'].isin([0, 1])
assert (reviewed_df.loc[_sup, 'resolved_sample_weight'].astype(float) > 0).all(), \
    "every supervised (0/1) row must have resolved_sample_weight > 0"
assert (~_erf_bool[_sup]).all(), "every supervised (0/1) row must have expert_review_required_final == False"

# --------------------------- counts ---------------------------
_counts = reviewed_df['resolved_periodontitis_label'].value_counts().sort_index().to_dict()
n_queue = int(_erf_bool.sum())
print("reviewed record-derived SILVER labels ->", MMD_REVIEWED_LABELS_CSV)
print(f"  patients: {len(reviewed_df)}  (unique ids: {reviewed_df['patient_id'].nunique()})")
print(f"  resolved_periodontitis_label: {_counts}  "
      f"(1=positive, 0=negative, -1=unresolved/expert-review)")
print(f"  review_resolution: {reviewed_df['review_resolution'].value_counts().to_dict()}")
print(f"  final expert-review queue size (expert_review_required_final): {n_queue}")
print("  NOTE: these are reviewed record-derived SILVER labels, NOT expert-confirmed "
      "gold labels; expert-confirmed gold labels still require expert CBCT review.")


reviewed record-derived SILVER labels -> /content/drive/MyDrive/Teeth_Segmentation_Classification/Preprocessed_Dataset/MMDental/logs/mmd_patient_labels_refined_reviewed.csv
  patients: 660  (unique ids: 660)
  resolved_periodontitis_label: {-1: 215, 0: 376, 1: 69}  (1=positive, 0=negative, -1=unresolved/expert-review)
  review_resolution: {'stage1_resolved': 410, 'expert_review_required': 215, 'record_resolved_positive': 25, 'record_resolved_negative': 10}
  final expert-review queue size (expert_review_required_final): 215
  NOTE: these are reviewed record-derived SILVER labels, NOT expert-confirmed gold labels; expert-confirmed gold labels still require expert CBCT review.


## 6B. REVIEWED STAGE-II RESOLVED-LABEL INTEGRATION (SILVER LABELS)

In [10]:
# --- Stage-II reviewed/resolved silver-label integration ---------------------
# Loads the reviewed record-derived labels (mmd_patient_labels_refined_reviewed.csv)
# from the two-stage adjudication. These REPLACE the weak record-derived labels as
# the Stage-II classification target. The weak labels are retained for audit only.
#
# Label semantics:  1 = resolved periodontitis-positive
#                    0 = resolved periodontitis-negative
#                   -1 = unresolved / expert-review-required (excluded from loss)
#
# These are REVIEWED RECORD-DERIVED SILVER labels, NOT expert-confirmed strong
# ground truth. Gold/strong ground truth still requires expert CBCT review.

# ---- new Stage-II artifact paths (label / manifest / split / CV only) ----
def _resolve_reviewed_csv():
    env = os.getenv("MMD_REVIEWED_LABELS_CSV")
    candidates = [pathlib.Path(env)] if env else []
    candidates += [
        MMDENTAL_RAW / 'mmd_patient_labels_refined_reviewed.csv',
        PROJECT_ROOT / 'mmd_patient_labels_refined_reviewed.csv',
        MMD_LOG_DIR / 'mmd_patient_labels_refined_reviewed.csv',
        MMD_MANIFEST_DIR / 'mmd_patient_labels_refined_reviewed.csv',
        pathlib.Path('mmd_patient_labels_refined_reviewed.csv'),
    ]
    for c in candidates:
        if c and c.exists():
            return c
    raise FileNotFoundError(
        "mmd_patient_labels_refined_reviewed.csv not found. Set MMD_REVIEWED_LABELS_CSV, "
        "or place it under MMDental/, the project root, or the manifests/logs dir. "
        f"Searched: {[str(c) for c in candidates]}")

REVIEWED_LABELS_CSV        = _resolve_reviewed_csv()
MMD_WEAK_LABELS_CSV        = MMD_LOG_DIR / 'mmd_patient_labels_weak.csv'
MANIFEST_STAGE2_RESOLVED   = MMD_MANIFEST_DIR / 'manifest_stage2_resolved.csv'
MANIFEST_STAGE2_SUPERVISED = MMD_MANIFEST_DIR / 'manifest_stage2_supervised.csv'
MANIFEST_STAGE2_UNRESOLVED = MMD_MANIFEST_DIR / 'manifest_stage2_unresolved.csv'
CV_STAGE2_RESOLVED         = MMD_MANIFEST_DIR / 'cv_stage2_resolved.csv'
SPLITS_STAGE2_RESOLVED     = MMD_MANIFEST_DIR / 'splits_stage2_resolved.csv'

# ---- load + validate the reviewed/resolved labels ----
REVIEWED_REQUIRED_COLS = [
    'patient_id', 'resolved_periodontitis_label', 'resolved_sample_weight',
    'expert_review_required_final', 'review_resolution', 'resolved_label_state',
    'resolved_confidence', 'resolution_reason', 'evidence_text', 'codes_found',
    'original_binary_label', 'refined_periodontitis_label', 'label_state',
]
reviewed_labels = pd.read_csv(REVIEWED_LABELS_CSV, encoding='utf-8-sig')
_missing = [c for c in REVIEWED_REQUIRED_COLS if c not in reviewed_labels.columns]
if _missing:
    raise ValueError(f"{REVIEWED_LABELS_CSV.name} missing required columns: {_missing}")

reviewed_labels['patient_id'] = reviewed_labels['patient_id'].astype(str).str.strip()

def _normalise_bool(series):
    if series.dtype == bool:
        return series
    return series.map(lambda v: str(v).strip().lower() in ('true', '1', '1.0', 'yes', 't'))
reviewed_labels['expert_review_required_final'] = _normalise_bool(
    reviewed_labels['expert_review_required_final'])

reviewed_labels['resolved_periodontitis_label'] = (
    pd.to_numeric(reviewed_labels['resolved_periodontitis_label'], errors='raise').astype(int))
reviewed_labels['resolved_sample_weight'] = (
    pd.to_numeric(reviewed_labels['resolved_sample_weight'], errors='raise').astype(float))

# ---- hard invariants on the label file itself ----
assert reviewed_labels['patient_id'].is_unique, "reviewed labels: patient_id not unique"
assert reviewed_labels['resolved_periodontitis_label'].isin([-1, 0, 1]).all(), \
    "resolved_periodontitis_label must be one of -1, 0, 1"
_unres = reviewed_labels['resolved_periodontitis_label'] == -1
assert (reviewed_labels.loc[_unres, 'resolved_sample_weight'] == 0.0).all(), \
    "every unresolved (-1) case must have resolved_sample_weight == 0.0"
assert (reviewed_labels['resolved_sample_weight'] >= 0.0).all(), "sample weights must be >= 0"

# ---- slim, collision-safe view merged into the manifest later ----
# Reviewed columns that would clash with legacy weak-label manifest columns
# (label_source / label_confidence / codes_found) are renamed so both the audit
# columns and the reviewed columns survive side by side.
reviewed_slim = reviewed_labels.rename(columns={
    'evidence_text': 'reviewed_evidence_text',
    'codes_found': 'reviewed_codes_found',
    'original_binary_label': 'weak_original_binary_label',
    'refined_periodontitis_label': 'stage1_refined_label',
    'label_state': 'stage1_label_state',
    'previous_label_state': 'stage1_previous_label_state',
})[[
    'patient_id', 'resolved_periodontitis_label', 'resolved_sample_weight',
    'expert_review_required_final', 'review_resolution', 'resolved_label_state',
    'resolved_confidence', 'resolution_reason',
    'reviewed_evidence_text', 'reviewed_codes_found',
    'weak_original_binary_label', 'stage1_refined_label', 'stage1_label_state',
]]

print(f"reviewed/resolved labels -> {REVIEWED_LABELS_CSV}  ({len(reviewed_labels)} rows)")
print("resolved_periodontitis_label:",
      reviewed_labels['resolved_periodontitis_label'].value_counts().sort_index().to_dict())
print("expert_review_required_final:",
      int(reviewed_labels['expert_review_required_final'].sum()), "flagged /", len(reviewed_labels))


reviewed/resolved labels -> /content/drive/MyDrive/Teeth_Segmentation_Classification/Preprocessed_Dataset/MMDental/logs/mmd_patient_labels_refined_reviewed.csv  (660 rows)
resolved_periodontitis_label: {-1: 215, 0: 376, 1: 69}
expert_review_required_final: 215 flagged / 660


## 7. IMAGING COHORT DISCOVERY AND MATCHING


In [11]:
def discover_nifti_files(raw_root):
    rows = []
    for path in sorted(raw_root.glob('*/*.nii.gz')):
        fid = path.name.replace('.nii.gz', '')
        rows.append({'patient_id': normalise_patient_id(fid), 'nifti_path': str(path),
                     'nifti_file_id': fid, 'nifti_parent_dir': path.parent.name})
    imaging = pd.DataFrame(rows)
    if len(imaging):
        dup = imaging['patient_id'][imaging['patient_id'].duplicated()].tolist()
        if dup:
            raise RuntimeError(f"duplicate imaging patient IDs: {dup[:20]}")
    return imaging

if HAVE_IMAGES:
    imaging = discover_nifti_files(MMDENTAL_RAW)
    cbct_ids = set(imaging["patient_id"].astype(str))

    patient_label_table["has_cbct"] = (
        patient_label_table["Filename"].astype(str).isin(cbct_ids)
    )
    patient_label_table["binary_label"] = (
        patient_label_table["binary_label_clean"]
    )

    cohort = imaging.merge(
        patient_label_table,
        left_on="patient_id",
        right_on="Filename",
        how="left",
        validate="one_to_one",
    ).copy()

    missing_records = cohort["Filename"].isna()
    if missing_records.any():
        log.warning(
            "%d imaging patients have no clinical record",
            int(missing_records.sum()),
        )

    cohort["Filename"] = cohort["patient_id"]

    record_ids = set(records_clean["patient_id"].astype(str))

    print(
        f"CBCT volumes: {len(imaging)} | "
        f"with records: {len(cbct_ids & record_ids)} | "
        f"records w/o imaging: {len(record_ids - cbct_ids)}"
    )

    show_counts(
        cohort[cohort["has_cbct"] == True],
        "CBCT IMAGING COHORT",
    )

    assert cohort["patient_id"].is_unique, (
        "Imaging cohort has duplicate patients"
    )

    assert int(patient_label_table["has_cbct"].sum()) == len(
        cbct_ids & record_ids
    ), (
        "The number of record patients marked has_cbct does not match "
        "the number of matched imaging patients"
    )

else:
    log.warning(
        "No CBCT volumes available; skipping imaging cohort, "
        "preprocessing, geometry, manifest, CV, and QA."
    )

    patient_label_table["has_cbct"] = pd.NA
    patient_label_table["binary_label"] = (
        patient_label_table["binary_label_clean"]
    )

    imaging = pd.DataFrame(
        columns=[
            "patient_id",
            "nifti_path",
            "nifti_file_id",
            "nifti_parent_dir",
        ]
    )
    cohort = patient_label_table.iloc[0:0].copy()

patient_label_table.to_csv(
    MMD_LABELS_CSV,
    index=False,
)

print(
    f"Saved legacy weak patient-level labels: "
    f"{MMD_LABELS_CSV} ({len(patient_label_table)} rows)"
)


CBCT volumes: 403 | with records: 403 | records w/o imaging: 257

=== CBCT IMAGING COHORT | n=403 ===
audit_subgroup:
audit_subgroup
OTHER_NONPERIODONTAL     242
GINGIVITIS_ONLY           40
APICAL_ONLY               40
PERIODONTITIS             28
OTHER_PERIODONTAL         24
NO_RECORDED_SIGNAL        10
APICAL_AND_GINGIVITIS      7
PERIO_ENDO_AMBIGUOUS       6
PERICORONAL                6

reliability_tier:
reliability_tier
0.25     30
0.50     10
0.75     49
1.00    314

Dataset A: 28 pos / 335 neg / 40 excluded | prevalence 7.7%
Dataset B: 53 pos / 335 neg / 15 excluded | prevalence 13.7%
Dataset C: POS 28 / NEG 335 / UNLABELED 40
Saved legacy weak patient-level labels: /content/drive/MyDrive/Teeth_Segmentation_Classification/Preprocessed_Dataset/MMDental/logs/mmd_patient_labels.csv (660 rows)


## 8-9. DUAL-SPACE CBCT PREPROCESSING WITH GEOMETRY SIDECARS


In [12]:
# ---- geometry helpers (verified on synthetic non-identity volumes) ----
def crop_pad_offsets(src_shape, target_shape):
    """Per numpy axis (z,y,x): (off, (pad_before, pad_after)); off = source index of output voxel 0."""
    offs, pads = [], []
    for s, t in zip(src_shape, target_shape):
        pb = max(0, (t - s) // 2); pa = max(0, t - s - (t - s) // 2)
        start = max(0, ((s + pb + pa) - t) // 2)
        offs.append(start - pb); pads.append((pb, pa))
    return offs, pads

def origin_after_crop_pad(origin0, spacing_xyz, direction_flat, off_zyx):
    D = np.array(direction_flat, float).reshape(3, 3)
    off_xyz = np.array([off_zyx[2], off_zyx[1], off_zyx[0]], float)
    return (np.array(origin0, float) + D @ (np.array(spacing_xyz, float) * off_xyz)).tolist()

def image_metadata(image):
    size = image.GetSize()

    return {
        "origin": tuple(float(value) for value in image.GetOrigin()),
        "direction": tuple(
            float(value) for value in image.GetDirection()
        ),
        "spacing": tuple(float(value) for value in image.GetSpacing()),
        "shape": (
            int(size[2]),
            int(size[1]),
            int(size[0]),
        ),
    }


def load_file_volume(path):
    source_image = sitk.ReadImage(str(path))
    source = image_metadata(source_image)

    reoriented_image = sitk.DICOMOrient(source_image, "LPS")
    reoriented = image_metadata(reoriented_image)

    return {
        "array": sitk.GetArrayFromImage(reoriented_image),
        "source": source,
        "reoriented": reoriented,
    }

def resample_to_spacing(arr, src_spacing, target_spacing, is_label=False, origin=None, direction=None):
    img = sitk.GetImageFromArray(arr)
    img.SetSpacing(tuple(float(s) for s in src_spacing))
    if origin is not None: img.SetOrigin(tuple(float(v) for v in origin))
    if direction is not None: img.SetDirection(tuple(float(v) for v in direction))
    src = img.GetSize()
    new = [max(1, int(round(src[i] * src_spacing[i] / target_spacing[i]))) for i in range(3)]
    r = sitk.ResampleImageFilter()
    r.SetOutputSpacing(tuple(float(v) for v in target_spacing)); r.SetSize(new)
    r.SetOutputOrigin(img.GetOrigin()); r.SetOutputDirection(img.GetDirection())
    r.SetInterpolator(sitk.sitkNearestNeighbor if is_label else sitk.sitkLinear)
    out = r.Execute(img)
    return sitk.GetArrayFromImage(out), out.GetOrigin(), out.GetDirection(), out.GetSpacing()

def crop_or_pad_center(arr, target, pad_value=0.0):
    pad = [(max(0,(t-s)//2), max(0,t-s-(t-s)//2)) for s,t in zip(arr.shape, target)]
    if any(b or e for b,e in pad):
        arr = np.pad(arr, pad, mode='constant', constant_values=pad_value)
    starts = [max(0,(c-t)//2) for c,t in zip(arr.shape, target)]
    return arr[tuple(slice(s,s+t) for s,t in zip(starts, target))]

def normalize_file_intensity(arr, cfg=CFG, eps=1e-6, air_pct=5.0):
    arr = arr.astype(np.float32, copy=True)
    lo, hi = (np.percentile(arr, cfg.clip_percentiles) if cfg.use_percentile_clip else cfg.fixed_clip_range)
    if hi <= lo: return np.zeros_like(arr)
    np.clip(arr, lo, hi, out=arr)
    fg = arr > np.percentile(arr, air_pct)
    if fg.sum() < 100: return np.zeros_like(arr)
    return (arr - arr[fg].mean()) / (arr[fg].std() + eps)


def geometry_roundtrip_ok(res_origin, res_direction, res_spacing, res_shape,
                          cls_origin, cls_shape, offs, tol=1e-3):
    """classifier voxel -> physical -> continuous index in the resampled frame == idx + off."""
    R = sitk.Image(int(res_shape[2]), int(res_shape[1]), int(res_shape[0]), sitk.sitkFloat32)
    R.SetSpacing(tuple(float(s) for s in res_spacing)); R.SetOrigin(tuple(res_origin))
    R.SetDirection(tuple(res_direction))
    C = sitk.Image(int(cls_shape[2]), int(cls_shape[1]), int(cls_shape[0]), sitk.sitkFloat32)
    C.SetSpacing(tuple(float(s) for s in CFG.classifier_target_spacing)); C.SetOrigin(tuple(cls_origin))
    C.SetDirection(tuple(res_direction))
    off_xyz = np.array([offs[2], offs[1], offs[0]], float)
    for oz, oy, ox in [(0,0,0), (cls_shape[0]//2, cls_shape[1]//2, cls_shape[2]//2),
                       (cls_shape[0]-1, cls_shape[1]-1, cls_shape[2]-1)]:
        p = C.TransformIndexToPhysicalPoint((int(ox), int(oy), int(oz)))
        idx = np.array(R.TransformPhysicalPointToContinuousIndex(p))
        if np.abs(idx - (np.array([ox, oy, oz]) + off_xyz)).max() > tol:
            return False
    return True


def preprocess_one(patient_id, nifti_path, cfg=CFG):
    cls_path = MMD_IMG3D_DIR / f'{patient_id}.npy'
    tea_path = MMD_TEACHER_IMG_DIR / f'{patient_id}.npy'
    geo_path = MMD_GEOMETRY_DIR / f'{patient_id}_geometry.json'
    if cfg.skip_existing and cls_path.exists() and tea_path.exists() and geo_path.exists():
        return True, {'patient_id': patient_id, 'status': 'cached',
                      'classifier_image_path': str(cls_path), 'teacher_image_path': str(tea_path),
                      'geometry_path': str(geo_path)}
    try:
        vol = load_file_volume(pathlib.Path(nifti_path))

        # teacher space: resample only (variable shape), geometry from SITK
        tea_arr, tea_o, tea_d, tea_sp = resample_to_spacing(
            vol["array"],
            vol["reoriented"]["spacing"],
            cfg.teacher_target_spacing,
            origin=vol["reoriented"]["origin"],
            direction=vol["reoriented"]["direction"],
        )
        tea_norm = normalize_file_intensity(tea_arr, cfg)
        np.save(tea_path, tea_norm.astype(cfg.save_dtype_image))

        # classifier space: resample to 1mm then center crop/pad to fixed shape; derive origin
        res_arr, res_o, res_d, res_sp = resample_to_spacing(
            vol["array"],
            vol["reoriented"]["spacing"],
            cfg.classifier_target_spacing,
            origin=vol["reoriented"]["origin"],
            direction=vol["reoriented"]["direction"],
        )
        pad_value = float(res_arr.min())
        cls_arr = crop_or_pad_center(res_arr, cfg.classifier_target_shape, pad_value=pad_value)
        offs, pads = crop_pad_offsets(res_arr.shape, cfg.classifier_target_shape)
        cls_origin = origin_after_crop_pad(res_o, cfg.classifier_target_spacing, res_d, offs)
        cls_norm = normalize_file_intensity(cls_arr, cfg)
        np.save(cls_path, cls_norm.astype(cfg.save_dtype_image))

        # geometry QA
        assert tuple(cls_arr.shape) == tuple(cfg.classifier_target_shape), "classifier shape mismatch"
        assert np.isfinite(cls_norm).all() and np.isfinite(tea_norm).all(), "non-finite voxels"
        assert all(s > 0 for s in tea_sp) and all(s > 0 for s in cfg.classifier_target_spacing)
        if not geometry_roundtrip_ok(res_o, res_d, res_sp, res_arr.shape,
                                     cls_origin, cfg.classifier_target_shape, offs):
            raise RuntimeError("geometry round-trip failed")

        geo = {
            'patient_id': patient_id, 'source_image_path': str(nifti_path),
            'config_version': cfg.config_version, 'array_axis_order': 'zyx',
            'transformation_order': ['DICOMOrient_LPS', 'resample_to_spacing', 'center_crop_or_pad(classifier_only)'],
            'orientation': 'LPS',
            # top-level keys consumed by the Stage-I bridge (teacher-space):
            'teacher_spacing': [float(s) for s in tea_sp], 'origin': list(map(float, tea_o)),
            'direction': list(map(float, tea_d)), 'shape': [int(x) for x in tea_norm.shape],
            "source": {
                "origin": list(vol["source"]["origin"]),
                "direction": list(vol["source"]["direction"]),
                "spacing": list(vol["source"]["spacing"]),
                "shape": list(vol["source"]["shape"]),
            },
            "reoriented": {
                "origin": list(vol["reoriented"]["origin"]),
                "direction": list(vol["reoriented"]["direction"]),
                "spacing": list(vol["reoriented"]["spacing"]),
                "shape": list(vol["reoriented"]["shape"]),
                "orientation": "LPS",
            },
            'teacher': {'origin': list(map(float, tea_o)), 'direction': list(map(float, tea_d)),
                        'spacing': [float(s) for s in tea_sp], 'shape': [int(x) for x in tea_norm.shape],
                        'interpolation': 'linear'},
            'classifier': {'origin': list(map(float, cls_origin)), 'direction': list(map(float, res_d)),
                           'effective_spacing': [float(s) for s in cfg.classifier_target_spacing],
                           'shape': [int(x) for x in cfg.classifier_target_shape],
                           'resampled_shape_precrop': [int(x) for x in res_arr.shape],
                           'interpolation': 'linear',
                           'crop_offsets_zyx': [int(o) for o in offs],
                           'pad_zyx': [[int(a), int(b)] for a, b in pads],
                           'pad_value': pad_value},
        }
        with open(geo_path, 'w') as fh:
            json.dump(geo, fh, indent=2)

        return True, {'patient_id': patient_id, 'status': 'processed',
                      'classifier_image_path': str(cls_path), 'teacher_image_path': str(tea_path),
                      'geometry_path': str(geo_path)}
    except Exception as exc:
        return False, {'patient_id': patient_id, 'status': 'failed', 'error': repr(exc)}


if HAVE_IMAGES:
    rows, failed = [], []
    for _, r in tqdm.auto.tqdm(cohort.iterrows(), total=len(cohort), desc='dual-space preprocess'):
        ok, res = preprocess_one(str(r['patient_id']), r['nifti_path'])
        (rows if ok else failed).append(res); gc.collect()
    if failed:
        pd.DataFrame(failed).to_csv(MMD_LOG_DIR / 'mmd_preprocessing_errors.csv', index=False)
        raise RuntimeError(f"{len(failed)} volumes failed; see logs. first: {failed[:3]}")
    volume_manifest = pd.DataFrame(rows)
    volume_manifest.to_csv(MMD_MANIFEST_DIR / 'manifest_volume_preprocessing.csv', index=False)
    print(volume_manifest['status'].value_counts().to_string())
else:
    volume_manifest = pd.DataFrame(columns=['patient_id','classifier_image_path','teacher_image_path','geometry_path','status'])
    print("preprocessing skipped (no volumes)")


dual-space preprocess:   0%|          | 0/403 [00:00<?, ?it/s]

status
processed    403


## 10. IMAGE AND GEOMETRY QA


In [13]:
if HAVE_IMAGES and len(volume_manifest):
    sample = volume_manifest[volume_manifest['status'].isin(['processed','cached'])].sample(
        min(8, len(volume_manifest)), random_state=SEED)
    fig, axes = plt.subplots(2, 4, figsize=(18, 8))
    issues = []
    for ax, (_, row) in zip(axes.flat, sample.iterrows()):
        arr = np.load(row['classifier_image_path'])
        if arr.shape != tuple(CFG.classifier_target_shape): issues.append((row['patient_id'],'shape',arr.shape))
        if not np.isfinite(arr).all(): issues.append((row['patient_id'],'nonfinite'))
        if not (0.3 < arr.std() < 3.0): issues.append((row['patient_id'],'std',float(arr.std())))
        with open(row['geometry_path']) as fh: geo = json.load(fh)
        if any(s <= 0 for s in geo['teacher_spacing']): issues.append((row['patient_id'],'spacing'))
        ax.imshow(arr[arr.shape[0]//2], cmap='gray'); ax.axis('off')
        ax.set_title(f"{row['patient_id']} std={arr.std():.2f}", fontsize=9)
    plt.tight_layout(); plt.savefig(MMD_LOG_DIR / 'mmd_qa_slices.png', dpi=110); plt.close()
    assert not issues, f"QA issues: {issues[:10]}"
    print(f"[ok] image+geometry QA passed on {len(sample)} sampled volumes")
else:
    print("image QA skipped (no volumes)")


[ok] image+geometry QA passed on 8 sampled volumes


## 11. PEDIATRIC AND EDENTULOUS ELIGIBILITY FLAGS


In [14]:
if HAVE_IMAGES and len(cohort):
    age_lookup = records_clean.groupby('patient_id')['Age'].median()
    cohort['age_median'] = cohort['patient_id'].map(age_lookup)
    cohort['pediatric'] = cohort['age_median'].fillna(99) < CFG.pediatric_age_max
    # edentulous heuristic from record evidence only (label-independent text cue)
    dx_all = patient_records_clean.set_index('patient_id')['Diagnosis_all'] if 'Diagnosis_all' in patient_records_clean else pd.Series(dtype=str)
    edent_cue = cohort['patient_id'].map(
        lambda p: bool(re.search(r'edentulous|full denture|complete denture|whole[- ]mouth (loss|missing)|total tooth loss',
                                 str(dx_all.get(p, '')).lower())))
    cohort['likely_edentulous'] = edent_cue.fillna(False)
    cohort['classifier_eligible'] = True   # retain for classification unless documented otherwise
    cohort['biomarker_evaluable'] = ~cohort['likely_edentulous']
    cohort['biomarker_exclusion_reason'] = np.where(cohort['likely_edentulous'], 'likely_edentulous', '')
    print(f"pediatric: {int(cohort['pediatric'].sum())} | likely_edentulous: {int(cohort['likely_edentulous'].sum())} | "
          f"biomarker_evaluable: {int(cohort['biomarker_evaluable'].sum())}")
else:
    print("eligibility flags skipped (no volumes)")


pediatric: 10 | likely_edentulous: 0 | biomarker_evaluable: 403


## 12. CLEAN IMAGING MANIFEST


In [15]:
MANIFEST_COLUMNS = [
    'patient_id', 'classifier_image_path', 'teacher_image_path', 'geometry_path',
    'binary_label_clean', 'binary_label_expanded',
    'dataset_A_member', 'dataset_B_member', 'dataset_C_label',
    'apical_aux_label', 'audit_subgroup', 'reliability_tier', 'sample_weight',
    'label_source', 'label_confidence', 'reliable_pos', 'reliable_neg',
    'pediatric', 'likely_edentulous', 'classifier_eligible',
    'biomarker_evaluable', 'biomarker_exclusion_reason',
    'binary_label',   # back-compat alias = binary_label_clean
]

if HAVE_IMAGES and len(cohort):
    m = cohort.merge(volume_manifest[['patient_id','classifier_image_path','teacher_image_path','geometry_path']],
                     on='patient_id', how='inner', validate='one_to_one')
    # confidence string for back-compat with Stage-0/biomarker (record_label_quality/label_confidence)
    m['label_confidence'] = m['reliability_tier'].map({1.0:'high', 0.75:'medium', 0.5:'low', 0.25:'low'})
    # reliable subsets for Stage-0 0C, computed under the corrected taxonomy
    m['reliable_pos'] = (m['label_source'] == 'code:K05.3')
    m['reliable_neg'] = (m['label_source'] == 'record_negative_code')
    m['binary_label'] = m['binary_label_clean']
    manifest = m[MANIFEST_COLUMNS].copy()
    manifest.to_csv(MMD_MANIFEST_CSV, index=False)
    print(f"saved manifest: {MMD_MANIFEST_CSV}  rows={len(manifest)}")
    print(f"reliable_pos={int(manifest['reliable_pos'].sum())} | reliable_neg={int(manifest['reliable_neg'].sum())}")
    display(manifest.head(3))
else:
    manifest = pd.DataFrame(columns=MANIFEST_COLUMNS)
    print("manifest skipped (no volumes)")


# ============================================================================
# Stage-II resolved-label manifests  (reviewed silver labels are the target)
# ============================================================================
# The legacy weak-label columns above (binary_label_clean/expanded,
# dataset_A/B_member, sample_weight, label_source, reliability_tier, ...) are kept
# for AUDIT ONLY and must NOT be used as the Stage-II target. The Stage-II target
# is `resolved_periodontitis_label`, weighted by `resolved_sample_weight`.

# weak record-derived labels persisted for audit / back-compat (never the target)
patient_label_table.to_csv(MMD_WEAK_LABELS_CSV, index=False)
print(f"weak labels (audit only) -> {MMD_WEAK_LABELS_CSV} ({len(patient_label_table)} rows)")

if HAVE_IMAGES and len(manifest):
    manifest_stage2_resolved = manifest.merge(
        reviewed_slim,
        on="patient_id",
        how="left",
        validate="one_to_one",
    )

    # ---- hard checks ----
    _missing_review = manifest_stage2_resolved["resolved_periodontitis_label"].isna()
    if _missing_review.any():
        _ids = manifest_stage2_resolved.loc[_missing_review, "patient_id"].tolist()
        raise AssertionError(
            f"{int(_missing_review.sum())} imaged patient(s) have no reviewed/resolved "
            f"label row: {_ids[:20]}"
        )

    manifest_stage2_resolved["resolved_periodontitis_label"] = (
        manifest_stage2_resolved["resolved_periodontitis_label"].astype(int)
    )

    assert manifest_stage2_resolved["resolved_periodontitis_label"].isin([-1, 0, 1]).all(), (
        "resolved_periodontitis_label must be one of -1, 0, 1"
    )

    _u = manifest_stage2_resolved["resolved_periodontitis_label"] == -1
    assert (manifest_stage2_resolved.loc[_u, "resolved_sample_weight"] == 0.0).all(), (
        "unresolved (-1) imaged cases must have resolved_sample_weight == 0.0"
    )

    # ---- Stage-I classifier-space segmentation paths ----
    STAGE1_INFER_ROOT = PROJECT_ROOT / "Stage1_Inference_MMDental"
    STAGE1_CLASSIFIER_SEG_DIR = STAGE1_INFER_ROOT / "classifier_space_seg"

    manifest_stage2_resolved["seg4_path"] = manifest_stage2_resolved["patient_id"].map(
        lambda pid: str(STAGE1_CLASSIFIER_SEG_DIR / f"{pid}_4class.npy")
    )

    manifest_stage2_resolved["seg_fdi_path"] = manifest_stage2_resolved["patient_id"].map(
        lambda pid: str(STAGE1_CLASSIFIER_SEG_DIR / f"{pid}_fdi.npy")
    )

    manifest_stage2_resolved["seg4_exists"] = manifest_stage2_resolved["seg4_path"].map(
        lambda p: pathlib.Path(p).exists()
    )

    manifest_stage2_resolved["seg_fdi_exists"] = manifest_stage2_resolved["seg_fdi_path"].map(
        lambda p: pathlib.Path(p).exists()
    )

    print("Stage-I 4-class masks found:", int(manifest_stage2_resolved["seg4_exists"].sum()))
    print("Stage-I FDI/group masks found:", int(manifest_stage2_resolved["seg_fdi_exists"].sum()))

    # ---- supervised / unresolved partitions ----
    is_supervised = (
        manifest_stage2_resolved["resolved_periodontitis_label"].isin([0, 1])
        & (manifest_stage2_resolved["resolved_sample_weight"] > 0)
        & (~manifest_stage2_resolved["expert_review_required_final"].astype(bool))
    )

    manifest_stage2_supervised = manifest_stage2_resolved[is_supervised].copy()
    manifest_stage2_unresolved = manifest_stage2_resolved[~is_supervised].copy()

    assert set(manifest_stage2_supervised["resolved_periodontitis_label"].unique()) <= {0, 1}, (
        "supervised manifest must contain only labels 0 and 1"
    )

    _overlap = (
        set(manifest_stage2_supervised["patient_id"])
        & set(manifest_stage2_unresolved["patient_id"])
    )
    assert not _overlap, f"supervised/unresolved overlap: {sorted(_overlap)[:20]}"

    # ---- save after all columns are attached ----
    manifest_stage2_resolved.to_csv(MANIFEST_STAGE2_RESOLVED, index=False)
    manifest_stage2_supervised.to_csv(MANIFEST_STAGE2_SUPERVISED, index=False)
    manifest_stage2_unresolved.to_csv(MANIFEST_STAGE2_UNRESOLVED, index=False)

    n_pos = int((manifest_stage2_supervised["resolved_periodontitis_label"] == 1).sum())
    n_neg = int((manifest_stage2_supervised["resolved_periodontitis_label"] == 0).sum())

    print("=== Stage-II resolved-label class counts (imaged cohort) ===")
    print(f"  Total imaged cases:                  {len(manifest_stage2_resolved)}")
    print(f"  Supervised resolved cases:           {len(manifest_stage2_supervised)}")
    print(f"    positive (resolved==1):            {n_pos}")
    print(f"    negative (resolved==0):            {n_neg}")
    print(f"  Unresolved / expert-review-required: {len(manifest_stage2_unresolved)}")
    print(f"  saved: {MANIFEST_STAGE2_RESOLVED.name}, {MANIFEST_STAGE2_SUPERVISED.name}, "
          f"{MANIFEST_STAGE2_UNRESOLVED.name}")

else:
    manifest_stage2_resolved = pd.DataFrame()
    manifest_stage2_supervised = pd.DataFrame()
    manifest_stage2_unresolved = pd.DataFrame()
    print("Stage-II resolved manifests skipped (no volumes)")

saved manifest: /content/drive/MyDrive/Teeth_Segmentation_Classification/Preprocessed_Dataset/MMDental/manifests/manifest.csv  rows=403
reliable_pos=23 | reliable_neg=291


,patient_id,classifier_image_path,teacher_image_path,geometry_path,binary_label_clean,binary_label_expanded,dataset_A_member,dataset_B_member,dataset_C_label,apical_aux_label,...,label_source,label_confidence,reliable_pos,reliable_neg,pediatric,likely_edentulous,classifier_eligible,biomarker_evaluable,biomarker_exclusion_reason,binary_label
0,1,/content/drive/MyDrive/Teeth_Segmentation_Clas...,/content/drive/MyDrive/Teeth_Segmentation_Clas...,/content/drive/MyDrive/Teeth_Segmentation_Clas...,0,0,True,True,NEGATIVE,0,...,record_negative_code,high,False,True,False,False,True,True,,0
1,10,/content/drive/MyDrive/Teeth_Segmentation_Clas...,/content/drive/MyDrive/Teeth_Segmentation_Clas...,/content/drive/MyDrive/Teeth_Segmentation_Clas...,0,0,True,True,NEGATIVE,0,...,record_negative_code,high,False,True,False,False,True,True,,0
2,100,/content/drive/MyDrive/Teeth_Segmentation_Clas...,/content/drive/MyDrive/Teeth_Segmentation_Clas...,/content/drive/MyDrive/Teeth_Segmentation_Clas...,0,0,True,True,NEGATIVE,1,...,record_negative_code,high,False,True,False,False,True,True,,0


weak labels (audit only) -> /content/drive/MyDrive/Teeth_Segmentation_Classification/Preprocessed_Dataset/MMDental/logs/mmd_patient_labels_weak.csv (660 rows)
Stage-I 4-class masks found: 0
Stage-I FDI/group masks found: 0
=== Stage-II resolved-label class counts (imaged cohort) ===
  Total imaged cases:                  403
  Supervised resolved cases:           282
    positive (resolved==1):            34
    negative (resolved==0):            248
  Unresolved / expert-review-required: 121
  saved: manifest_stage2_resolved.csv, manifest_stage2_supervised.csv, manifest_stage2_unresolved.csv


In [16]:
STAGE1_GEOMETRY_AUDIT_CSV = (
    PROJECT_ROOT
    / "Stage1_Inference_MMDental"
    / "classifier_space_seg_geometry_audit_after_fix.csv"
)

if STAGE1_GEOMETRY_AUDIT_CSV.exists():
    audit = pd.read_csv(STAGE1_GEOMETRY_AUDIT_CSV)

    assert (audit["status"] == "ok").all(), "Stage-I geometry audit contains failed cases"

    if "same_shape" in audit.columns:
        assert audit["same_shape"].all(), "Some Stage-I classifier-space masks have shape mismatch"

    if "exported_vs_geometry_dice" in audit.columns:
        assert (audit["exported_vs_geometry_dice"] > 0.999).all(), (
            "Some Stage-I masks do not match geometry-based reconstruction"
        )

    pad_cols = [
        c for c in audit.columns
        if c.startswith("exported_") and c.endswith("_pad_fraction")
    ]

    for c in pad_cols:
        assert (audit[c] == 0.0).all(), f"Padding leakage found in {c}"

    print("[ok] Stage-I classifier-space geometry audit passed")
else:
    log.warning(
        "Stage-I geometry audit not found. Stage-II manifests can be created, "
        "but anatomy-guided training should not be considered locked until this audit passes."
    )

## 13. REPEATED PATIENT-LEVEL CROSS-VALIDATION

In [17]:
# Stage-II cross-validation is built ONLY from manifest_stage2_supervised, using
# `resolved_periodontitis_label` as the target. It never uses the old Dataset A/B
# weak labels, and unresolved (-1) cases are never placed in any fold (they carry
# weight 0 and are reserved for inference / expert review).

def assign_stage2_cv(sup_df, cfg=CFG, seed=SEED):
    sub = sup_df.copy()
    sub['resolved_periodontitis_label'] = sub['resolved_periodontitis_label'].astype(int)
    assert sub['resolved_periodontitis_label'].isin([0, 1]).all(), "supervised must be 0/1 only"

    # richer stratification on label x apical-aux when each stratum is large enough
    if 'apical_aux_label' in sub.columns:
        compound = sub.apply(
            lambda r: f"{int(r['resolved_periodontitis_label'])}|aux{int(r['apical_aux_label'])}",
            axis=1)
        if compound.value_counts().min() >= cfg.cv_n_splits:
            strat, mode = compound, 'resolved_label_x_apical_aux'
        else:
            strat, mode = sub['resolved_periodontitis_label'].astype(str), 'resolved_label_only'
    else:
        strat, mode = sub['resolved_periodontitis_label'].astype(str), 'resolved_label_only'

    min_class = int(sub['resolved_periodontitis_label'].value_counts().min())
    if min_class < cfg.cv_n_splits:
        raise ValueError(
            f"smallest supervised class ({min_class}) < cv_n_splits ({cfg.cv_n_splits}); "
            f"cannot build a stratified {cfg.cv_n_splits}-fold CV")

    rskf = sklearn.model_selection.RepeatedStratifiedKFold(
        n_splits=cfg.cv_n_splits, n_repeats=cfg.cv_n_repeats, random_state=seed)
    idx = sub.reset_index(drop=True)
    rows = []
    for split_i, (_, test_idx) in enumerate(rskf.split(idx, strat.values)):
        for j in test_idx:
            rows.append({'patient_id': idx.iloc[j]['patient_id'],
                         'resolved_periodontitis_label': int(idx.iloc[j]['resolved_periodontitis_label']),
                         'resolved_sample_weight': float(idx.iloc[j]["resolved_sample_weight"]),
                         'repeat': split_i // cfg.cv_n_splits,
                         'fold': split_i % cfg.cv_n_splits})
    return pd.DataFrame(rows), mode

def fold_role(fold, test_fold, n=CFG.cv_n_splits):
    if fold == test_fold: return 'test'
    if fold == (test_fold + 1) % n: return 'val'
    return 'train'

if HAVE_IMAGES and len(manifest_stage2_supervised):
    cv_stage2, cv_mode = assign_stage2_cv(manifest_stage2_supervised)
    n_sup = len(manifest_stage2_supervised)

    # patient-level integrity: each supervised patient assigned once per repeat, no leakage
    assert cv_stage2.groupby('repeat')['patient_id'].nunique().eq(n_sup).all(), \
        "each supervised patient must appear once per repeat"
    assert cv_stage2.groupby(['repeat', 'patient_id']).size().eq(1).all(), "per-repeat leakage"

    # unresolved cases must never appear in CV
    _unres_ids = set(manifest_stage2_unresolved['patient_id'])
    assert not (set(cv_stage2['patient_id']) & _unres_ids), \
        "unresolved / expert-review cases must not appear in Stage-II CV"

    cv_stage2.to_csv(CV_STAGE2_RESOLVED, index=False)

    rep0 = cv_stage2[cv_stage2['repeat'] == 0].copy()
    rep0['role'] = rep0['fold'].apply(lambda f: fold_role(f, 0))
    summ = rep0.groupby('role').agg(
        n=('patient_id', 'size'), pos=('resolved_periodontitis_label', 'sum'))
    print(f"Stage-II CV ({cv_mode}) | supervised members={n_sup} "
          f"| repeats={CFG.cv_n_repeats} x {CFG.cv_n_splits}-fold")
    print("repeat 0 roles:"); print(summ.to_string())
    print(f"saved CV assignments -> {CV_STAGE2_RESOLVED}  ({len(cv_stage2)} rows)")

    # canonical single split (repeat 0) for downstream convenience
    splits0 = rep0[['patient_id']].copy()
    splits0['split'] = rep0['fold'].apply(lambda f: fold_role(f, 0)).values
    splits0.to_csv(SPLITS_STAGE2_RESOLVED, index=False)
    print(f"saved splits -> {SPLITS_STAGE2_RESOLVED}  ({len(splits0)} rows)")

    # write the split back onto the full resolved manifest; unresolved -> excluded
    split_map = dict(zip(splits0['patient_id'], splits0['split']))
    manifest_stage2_resolved['split'] = (
        manifest_stage2_resolved['patient_id'].map(split_map).fillna('excluded_unresolved'))
    manifest_stage2_resolved.to_csv(MANIFEST_STAGE2_RESOLVED, index=False)
    print(f"updated {MANIFEST_STAGE2_RESOLVED.name} with Stage-II `split` column")
else:
    cv_stage2 = pd.DataFrame()
    print("Stage-II CV skipped (no volumes or no supervised cases)")


Stage-II CV (resolved_label_only) | supervised members=282 | repeats=5 x 5-fold
repeat 0 roles:
         n  pos
role           
test    57    7
train  168   20
val     57    7
saved CV assignments -> /content/drive/MyDrive/Teeth_Segmentation_Classification/Preprocessed_Dataset/MMDental/manifests/cv_stage2_resolved.csv  (1410 rows)
saved splits -> /content/drive/MyDrive/Teeth_Segmentation_Classification/Preprocessed_Dataset/MMDental/manifests/splits_stage2_resolved.csv  (282 rows)
updated manifest_stage2_resolved.csv with Stage-II `split` column


## 14. BIOMARKER-GROUNDED QA (DEFERRED UNTIL THE BIOMARKER EXISTS)


In [18]:
# Rich, biomarker-grounded QA generation for Qwen2.5-VL fine-tuning.
#
# Scientific guardrails (unchanged from the resolved-label contract):
#   * Ground-truth DISEASE QA (binary_classification / classification_explanation /
#     structured_periodontal_report / limitations) is generated ONLY for supervised
#     cases: resolved_periodontitis_label in {0,1} AND expert_review_required_final == False.
#   * Unresolved cases (resolved_periodontitis_label == -1 OR expert_review_required_final
#     == True) receive ONLY biomarker_evidence, expert_review_prompt, and
#     uncertainty_explanation - never a disease-classification answer.
#   * Every answer is image-grounded (image-derived bone-support biomarkers); a patient
#     with no usable biomarker fields produces no QA rows.
#
# These are REVIEWED RECORD-DERIVED SILVER labels, not expert-confirmed CBCT gold labels.

QA_STATUS_RICH_JSON = MMD_QA_DIR / 'qa_status_rich.json'
MMD_QA_JSONL_RICH   = MMD_QA_DIR / 'qa_pairs_rich.jsonl'

SILVER_LABEL_NOTE = (
    "These QA pairs are biomarker-grounded and use reviewed record-derived silver "
    "labels. They are not expert-confirmed CBCT gold-label explanations. True gold "
    "QA requires expert CBCT review."
)

def _biomarker_parts(r):
    """Readable parts + field names from the three image-derived biomarkers."""
    fields, parts = [], []
    if 'min_support' in r and pd.notna(r['min_support']):
        fields.append('min_support')
        parts.append(f"the worst-group supporting-bone fraction is {float(r['min_support']):.2f}")
    if 'n_affected_groups' in r and pd.notna(r['n_affected_groups']):
        fields.append('n_affected_groups')
        parts.append(f"{int(r['n_affected_groups'])} tooth group(s) show reduced supporting bone")
    if 'min_bone_level' in r and pd.notna(r['min_bone_level']):
        fields.append('min_bone_level')
        parts.append(f"the lowest crownward bone-level fraction is {float(r['min_bone_level']):.2f}")
    return fields, parts

def _base_row(r, seg4, seg_fdi, fields, summary):
    return {
        'patient_id': r['patient_id'],
        'volume_path': r['classifier_image_path'],
        'seg4_path': seg4,
        'seg_fdi_path': seg_fdi,
        'split': r.get('split', 'unassigned'),
        'resolved_periodontitis_label': int(r['resolved_periodontitis_label']),
        'resolved_label_state': r.get('resolved_label_state'),
        'resolved_confidence': (float(r['resolved_confidence'])
                                if pd.notna(r.get('resolved_confidence')) else None),
        'resolved_sample_weight': (float(r['resolved_sample_weight'])
                                   if pd.notna(r.get('resolved_sample_weight')) else None),
        'expert_review_required_final': bool(r['expert_review_required_final']),
        'biomarkers_used': ';'.join(fields),
        'biomarker_summary': summary,
        'source_manifest': str(MANIFEST_STAGE2_RESOLVED),
        'source_biomarker_csv': str(PER_PATIENT_BIOMARKER_CSV),
    }

def build_rich_qa(man_df, bio_df):
    j = man_df.merge(bio_df, on='patient_id', how='inner', suffixes=('', '_bio'))
    seg4_col   = 'seg4_path'    if 'seg4_path'    in j.columns else None
    segfdi_col = 'seg_fdi_path' if 'seg_fdi_path' in j.columns else None
    rows, patients_with_qa = [], set()

    for _, r in j.iterrows():
        if not bool(r.get('biomarker_evaluable', True)):
            continue
        fields, parts = _biomarker_parts(r)
        if not fields:                      # keep everything image-grounded
            continue
        summary = "; ".join(parts)
        seg4    = (r[seg4_col]   if seg4_col   and pd.notna(r[seg4_col])   else None)
        seg_fdi = (r[segfdi_col] if segfdi_col and pd.notna(r[segfdi_col]) else None)
        label = int(r['resolved_periodontitis_label'])
        supervised = (label in (0, 1)) and (not bool(r['expert_review_required_final']))
        label_usage = 'supervised_ground_truth' if supervised else 'review_or_inference_only'
        state = r.get('resolved_label_state')
        conf = r.get('resolved_confidence')
        conf_txt = f"{float(conf):.2f}" if pd.notna(conf) else "unavailable"
        reason = str(r.get('resolution_reason') or '').strip()

        def emit(qtype, question, answer, provenance, grounding):
            row = _base_row(r, seg4, seg_fdi, fields, summary)
            row.update({'question_type': qtype, 'question': question, 'answer': answer,
                        'label_usage': label_usage, 'answer_provenance': provenance,
                        'grounding': grounding})
            rows.append(row)
            patients_with_qa.add(r['patient_id'])

        # 1) biomarker evidence (image-only; both supervised and unresolved)
        emit('biomarker_evidence',
             "What image-derived periodontal supporting-bone evidence is visible in this CBCT volume?",
             (f"Based on the image-derived bone-support biomarker, {summary}. This is radiographic "
              f"supporting-bone evidence measured from the CBCT volume and is independent of the clinical record."),
             ';'.join(fields), 'image_biomarker_derived')

        if supervised:
            polarity = 'positive' if label == 1 else 'negative'

            # 2) binary classification
            if label == 1:
                cls_ans = (f"This case is labelled periodontitis-positive. The reviewed record-derived silver "
                           f"label is positive, consistent with the image-derived evidence that {summary}. This is "
                           f"a reviewed record-derived silver label, not an expert-confirmed CBCT diagnosis.")
            else:
                cls_ans = (f"This case is labelled periodontitis-negative. The reviewed record-derived silver "
                           f"label is negative; the image-derived evidence shows {summary}. This is a reviewed "
                           f"record-derived silver label, not an expert-confirmed CBCT diagnosis.")
            emit('binary_classification',
                 ("Based on the image-derived supporting-bone biomarkers and the reviewed record-derived label, "
                  "is this case periodontitis-positive or periodontitis-negative?"),
                 cls_ans, 'resolved_periodontitis_label;' + ';'.join(fields),
                 'image_biomarker_plus_resolved_silver_label')

            # 3) classification explanation
            if label == 1:
                exp_ans = (f"The case is classified periodontitis-positive because the reviewed adjudication "
                           f"resolved it to a positive state ('{state}') and the image-derived biomarkers indicate "
                           f"reduced supporting bone: {summary}. This label is a reviewed record-derived silver "
                           f"label; definitive confirmation requires expert CBCT review.")
            else:
                exp_ans = (f"The case is classified periodontitis-negative because the reviewed adjudication "
                           f"resolved it to a negative state ('{state}') with no confirming periodontal evidence, "
                           f"and the image-derived biomarkers are reported as {summary}. This label is a reviewed "
                           f"record-derived silver label; definitive confirmation requires expert CBCT review.")
            emit('classification_explanation',
                 f"Why is this case classified as periodontitis-{polarity}?",
                 exp_ans, 'resolved_periodontitis_label;resolved_label_state;' + ';'.join(fields),
                 'image_biomarker_plus_resolved_silver_label')

            # 4) structured periodontal report
            report = ["Structured periodontal supporting-bone report:"]
            if 'min_support' in fields:
                report.append(f"- Worst-group supporting-bone fraction: {float(r['min_support']):.2f}")
            if 'n_affected_groups' in fields:
                report.append(f"- Tooth groups with reduced supporting bone: {int(r['n_affected_groups'])}")
            if 'min_bone_level' in fields:
                report.append(f"- Lowest crownward bone-level fraction: {float(r['min_bone_level']):.2f}")
            report.append(f"- Resolved label state: {state} (label={label}, {polarity})")
            report.append(f"- Label confidence: {conf_txt}")
            report.append(f"- Note: {SILVER_LABEL_NOTE}")
            emit('structured_periodontal_report',
                 ("Provide a structured periodontal supporting-bone report for this CBCT case, including worst "
                  "bone support, affected tooth groups, lowest bone-level fraction, label state, and confidence."),
                 "\n".join(report),
                 'resolved_periodontitis_label;resolved_label_state;resolved_confidence;' + ';'.join(fields),
                 'image_biomarker_plus_resolved_silver_label')

            # 5) limitations
            emit('limitations',
                 "What are the limitations of this classification and its supporting evidence?",
                 (f"Limitations: the label used here is a reviewed record-derived silver label produced by "
                  f"automated adjudication of the clinical record, not an expert-confirmed CBCT gold label. The "
                  f"supporting evidence is limited to image-derived bone-support biomarkers ({', '.join(fields)}) "
                  f"and does not include full periodontal charting or probing depths. The resolved confidence is "
                  f"{conf_txt}. {SILVER_LABEL_NOTE}"),
                 'resolved_confidence;' + ';'.join(fields),
                 'image_biomarker_plus_resolved_silver_label')
        else:
            # unresolved: review + uncertainty only (NO disease classification)
            emit('expert_review_prompt',
                 "Should this case be sent for expert CBCT review before assigning a periodontitis label?",
                 (f"Yes. This case is unresolved (resolved label = -1, expert_review_required_final = True) and has "
                  f"no reliable record-derived disease label. The image-derived biomarkers ({summary}) can inform "
                  f"review prioritization, but a periodontitis-positive or -negative determination requires expert "
                  f"CBCT review. No ground-truth disease label is asserted."),
                 'resolved_periodontitis_label;expert_review_required_final;' + ';'.join(fields),
                 'review_only_no_ground_truth_label')

            reason_txt = (f" The adjudication reason was: {reason}." if reason else "")
            emit('uncertainty_explanation',
                 "Why can't a periodontitis-positive or -negative label be assigned to this case from the record?",
                 (f"The reviewed adjudication left this case in state '{state}' with expert review required, because "
                  f"the clinical record was ambiguous, conflicting, gingivitis-only, apical-only, or otherwise "
                  f"insufficient to confirm marginal periodontitis.{reason_txt} Therefore no supervised disease "
                  f"label is assigned. The image-derived biomarkers ({summary}) are provided for review only, not "
                  f"as ground truth."),
                 'resolved_label_state;expert_review_required_final;resolution_reason;' + ';'.join(fields),
                 'review_only_no_ground_truth_label')

    return pd.DataFrame(rows), patients_with_qa

if HAVE_IMAGES and len(manifest_stage2_resolved):
    qa_source = manifest_stage2_resolved.copy()
    qa_source['patient_id'] = qa_source['patient_id'].astype(str).str.strip()

    if PER_PATIENT_BIOMARKER_CSV.exists():
        bio_df = pd.read_csv(PER_PATIENT_BIOMARKER_CSV)
        bio_df['patient_id'] = bio_df['patient_id'].astype(str).str.strip()
        usable = [c for c in ('min_support', 'n_affected_groups', 'min_bone_level') if c in bio_df.columns]
        if not usable:
            raise ValueError("Biomarker table contains none of: min_support, n_affected_groups, min_bone_level")

        manifest_ids, biomarker_ids = set(qa_source['patient_id']), set(bio_df['patient_id'])
        print("Imaged patients missing from biomarker table:", len(manifest_ids - biomarker_ids),
              sorted(manifest_ids - biomarker_ids)[:10])

        qa_rich, patients_with_qa = build_rich_qa(qa_source, bio_df)

        # ---- guardrail assertions ----
        _gt_types = {'binary_classification', 'classification_explanation',
                     'structured_periodontal_report', 'limitations'}
        if len(qa_rich):
            _bad = qa_rich[(qa_rich['label_usage'] == 'review_or_inference_only')
                           & (qa_rich['question_type'].isin(_gt_types))]
            assert _bad.empty, "unresolved cases must not receive ground-truth disease QA"
            assert (qa_rich.loc[qa_rich['question_type'].isin(_gt_types), 'grounding']
                    == 'image_biomarker_plus_resolved_silver_label').all(), "disease QA must carry silver-label grounding"
            _unres_ids = set(manifest_stage2_unresolved['patient_id'])
            _leaked = _unres_ids & set(
                qa_rich.loc[qa_rich['label_usage'] == 'supervised_ground_truth', 'patient_id'])
            assert not _leaked, f"unresolved patients tagged supervised_ground_truth: {sorted(_leaked)[:10]}"

        with open(MMD_QA_JSONL_RICH, 'w', encoding='utf-8') as fh:
            for _, row in qa_rich.iterrows():
                fh.write(json.dumps(row.to_dict(), ensure_ascii=False) + '\n')

        n_gt = int((qa_rich['label_usage'] == 'supervised_ground_truth').sum()) if len(qa_rich) else 0
        n_rev = int((qa_rich['label_usage'] == 'review_or_inference_only').sum()) if len(qa_rich) else 0
        by_type = qa_rich['question_type'].value_counts().to_dict() if len(qa_rich) else {}
        by_grounding = qa_rich['grounding'].value_counts().to_dict() if len(qa_rich) else {}

        qa_status_rich = {
            'status': 'generated',
            'n_patients_with_qa': int(len(patients_with_qa)),
            'n_qa': int(len(qa_rich)),
            'n_supervised_ground_truth': n_gt,
            'n_review_or_inference_only': n_rev,
            'n_by_question_type': by_type,
            'n_by_grounding': by_grounding,
            'grounding': ('image_biomarker_derived | image_biomarker_plus_resolved_silver_label | '
                          'review_only_no_ground_truth_label'),
            'label_semantics': ('reviewed record-derived silver labels '
                                '(1=positive, 0=negative, -1=unresolved/expert-review); '
                                'not expert-confirmed CBCT gold labels'),
            'source_manifest': str(MANIFEST_STAGE2_RESOLVED),
            'biomarker_csv': str(PER_PATIENT_BIOMARKER_CSV),
            'note': SILVER_LABEL_NOTE,
        }
        json.dump(qa_status_rich, open(QA_STATUS_RICH_JSON, 'w'), indent=2)
        print(f"[ok] rich QA: {len(qa_rich)} rows over {len(patients_with_qa)} patients "
              f"({n_gt} supervised-ground-truth, {n_rev} review/inference-only) -> {MMD_QA_JSONL_RICH}")
        print("  by question_type:", by_type)
        print(f"  NOTE: {SILVER_LABEL_NOTE}")
    else:
        skeleton = qa_source[['patient_id', 'classifier_image_path',
                              'resolved_periodontitis_label', 'expert_review_required_final']].copy()
        skeleton['split'] = qa_source['split'] if 'split' in qa_source.columns else 'unassigned'
        skeleton['qa_role'] = np.where(
            skeleton['resolved_periodontitis_label'].isin([0, 1])
            & (~skeleton['expert_review_required_final'].astype(bool)),
            'ground_truth_eligible', 'review_or_inference_only')
        skeleton.to_csv(MMD_QA_DIR / 'qa_skeleton_rich.csv', index=False)
        json.dump({'status': 'deferred',
                   'reason': 'per_patient_biomarkers.csv not found; rich QA must be image-grounded',
                   'expected_biomarker_csv': str(PER_PATIENT_BIOMARKER_CSV),
                   'skeleton_rows': int(len(skeleton)),
                   'ground_truth_eligible': int((skeleton['qa_role'] == 'ground_truth_eligible').sum()),
                   'label_semantics': ('reviewed record-derived silver labels; '
                                       'not expert-confirmed CBCT gold labels'),
                   'note': SILVER_LABEL_NOTE}, open(QA_STATUS_RICH_JSON, 'w'), indent=2)
        log.warning("Rich QA DEFERRED: biomarker table absent. Wrote qa_skeleton_rich.csv (no answers). "
                    "Re-run after the Stage-II biomarker stage.")
else:
    print("Rich QA skipped (no volumes)")


## 15. FINAL ARTIFACT CHECKLIST


In [19]:
artifacts = {
    'weak labels (audit only)':        MMD_WEAK_LABELS_CSV,
    'reviewed/resolved labels':        REVIEWED_LABELS_CSV,
    'Stage-II manifest (resolved)':    MANIFEST_STAGE2_RESOLVED,
    'Stage-II manifest (supervised)':  MANIFEST_STAGE2_SUPERVISED,
    'Stage-II manifest (unresolved)':  MANIFEST_STAGE2_UNRESOLVED,
    'Stage-II CV assignments':         CV_STAGE2_RESOLVED,
    'Stage-II splits (repeat 0)':      SPLITS_STAGE2_RESOLVED,
    'rich QA pairs (Qwen2.5-VL)':      MMD_QA_JSONL_RICH,
    'rich QA status':                  QA_STATUS_RICH_JSON,
    'classifier-space image dir':      MMD_IMG3D_DIR,
    'teacher-space image dir':         MMD_TEACHER_IMG_DIR,
    'geometry sidecar dir':            MMD_GEOMETRY_DIR,
    # legacy (kept for audit / Stage-0 back-compat only)
    'legacy weak manifest':            MMD_MANIFEST_CSV,
    'legacy patient labels':           MMD_LABELS_CSV,
}
print("=== artifact checklist ===")
for name, path in artifacts.items():
    print(f"  [{'ok' if pathlib.Path(path).exists() else '--'}] {name}: {path}")

print("\n=== Stage-II downstream contract ===")
print("  - Stage-II classifier MUST train from manifest_stage2_supervised.csv")
print("  - Target column:        resolved_periodontitis_label   (1=positive, 0=negative)")
print("  - Sample-weight column: resolved_sample_weight")
print("  - Exclude / mask ALL cases where resolved_periodontitis_label == -1")
print("    or expert_review_required_final == True (these carry weight 0).")
print("  - Use manifest_stage2_unresolved.csv ONLY for inference, review")
print("    prioritization, or expert CBCT adjudication - never as supervised targets.")
print("  - Cross-validation: cv_stage2_resolved.csv / splits_stage2_resolved.csv")
print("    (repeated stratified 5-fold on resolved_periodontitis_label, patient-level).")
print("  - Stage-I geometry bridge unchanged: teacher_geometry/{Filename}_geometry.json")
print("  - VLM (Qwen2.5-VL) fine-tuning QA: qa_pairs_rich.jsonl (biomarker-grounded;")
print("    supervised cases carry silver-label disease QA, unresolved cases carry")
print("    review/uncertainty QA only). Status: qa_status_rich.json.")
print("  - Legacy weak columns (binary_label_clean/expanded, dataset_A/B_member,")
print("    sample_weight, label_source, reliability_tier) are AUDIT ONLY.")
print("\nScientific note: these are REVIEWED RECORD-DERIVED SILVER labels, not")
print("expert-confirmed strong ground truth. Gold/strong ground truth still")
print("requires expert CBCT review.")


=== artifact checklist ===
  [ok] weak labels (audit only): /content/drive/MyDrive/Teeth_Segmentation_Classification/Preprocessed_Dataset/MMDental/logs/mmd_patient_labels_weak.csv
  [ok] reviewed/resolved labels: /content/drive/MyDrive/Teeth_Segmentation_Classification/Preprocessed_Dataset/MMDental/logs/mmd_patient_labels_refined_reviewed.csv
  [ok] Stage-II manifest (resolved): /content/drive/MyDrive/Teeth_Segmentation_Classification/Preprocessed_Dataset/MMDental/manifests/manifest_stage2_resolved.csv
  [ok] Stage-II manifest (supervised): /content/drive/MyDrive/Teeth_Segmentation_Classification/Preprocessed_Dataset/MMDental/manifests/manifest_stage2_supervised.csv
  [ok] Stage-II manifest (unresolved): /content/drive/MyDrive/Teeth_Segmentation_Classification/Preprocessed_Dataset/MMDental/manifests/manifest_stage2_unresolved.csv
  [ok] Stage-II CV assignments: /content/drive/MyDrive/Teeth_Segmentation_Classification/Preprocessed_Dataset/MMDental/manifests/cv_stage2_resolved.csv
  [ok]